In [24]:
"""
=============================================================================
텍스트 기반 우울증 감지 - 전처리 파이프라인 (MentalBERT)
=============================================================================

개선사항:
  1. MentalBERT 사용 (우울증/정신건강 도메인 특화)
  2. Filler words 제거 (um, uh, hmm 등)
  3. NLTK 불용어 제거
  4. 더 정교한 텍스트 정제

방법론: MentalBERT Embeddings + Linguistic Features
특징 구성: [MentalBERT(768) + Q-type(32)] = 800차원
"""

import torch
from transformers import AutoTokenizer, AutoModel
import pandas as pd
import numpy as np
import pickle
import os
from tqdm import tqdm
import re
from collections import defaultdict
import warnings
import nltk
from nltk.corpus import stopwords

warnings.filterwarnings('ignore')

# =============================================================================
# NLTK 데이터 다운로드 (최초 1회)
# =============================================================================
try:
    nltk.data.find('corpora/stopwords')
except LookupError:
    print("⏳ NLTK stopwords 다운로드 중...")
    nltk.download('stopwords', quiet=True)

# =============================================================================
# 설정
# =============================================================================
BASE_PATH = r"D:\depression_dataset(DAIC-WOZ)"

# 예외 처리할 참가자
EXCEPTION_NUMBER = [
    '451', '458', '480'  # Ellie 발화 누락 (3명)
]

# 출력 파일
OUTPUT_PATH = os.path.join(BASE_PATH, "preprocessed_text_mentalbert_dataset.pkl")

# MentalBERT 설정
# MentalBERT: 정신건강 관련 Reddit 데이터로 사전학습된 BERT 모델
MENTAL_BERT_MODEL = 'mental/mental-bert-base-uncased'
MAX_LENGTH = 128

# Filler words 및 불필요한 표현
FILLER_WORDS = {
    # 기본 fillers
    'um', 'uh', 'hmm', 'mhm', 'huh', 'ah', 'oh', 'mm', 'mmm',
    'er', 'erm', 'uhm', 'umm', 'erm',
    
    # 반복적인 표현
    'like', 'yeah', 'yes', 'yep', 'nope', 'nah',
    
    # 불필요한 표현
    'know', 'los angeles', 'nan', 'na',
    
    # 추가 fillers
    'basically', 'actually', 'literally', 'sort of', 'kind of',
    'i mean', 'you know', 'well',
}

# NLTK 불용어
STOP_WORDS = set(stopwords.words('english'))

# 정신건강 관련 키워드는 보존 (불용어에서 제외)
PRESERVE_WORDS = {
    'not', 'no', 'never', 'nothing', 'nobody', 'none',  # 부정어
    'but', 'however', 'although',  # 전환어
    'very', 'really', 'quite', 'too', 'more', 'most',  # 강조어
    'down', 'up', 'off', 'out',  # 감정 관련 전치사
}

# 최종 불용어 = NLTK 불용어 - 보존 단어
STOPWORDS_TO_REMOVE = STOP_WORDS - PRESERVE_WORDS

# Q-type 매핑
Q_TYPE_MAPPING = {
    'casual': 0,
    'background': 1,
    'emotional': 2,
    'clinical': 3,
    'other': 4
}

Q_TYPE_SIMPLIFICATION = {
    'small talk': 'casual',
    'preference': 'casual',
    'open-ended encouragement': 'casual',
    
    'daily habits / lifestyle': 'background',
    'social / family / relationship': 'background',
    'self-perception / personality': 'background',
    
    'emotion / mood': 'emotional',
    
    'depression symptoms direct': 'clinical',
    
    'other': 'other'
}

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

print(f"{'='*70}")
print(f"📝 텍스트 전처리 시작 (MentalBERT 기반)")
print(f"{'='*70}")
print(f"Device: {DEVICE}")
print(f"Model: {MENTAL_BERT_MODEL}")
print(f"Filler words: {len(FILLER_WORDS)}개")
print(f"Stopwords: {len(STOPWORDS_TO_REMOVE)}개")
print(f"{'='*70}\n")


# =============================================================================
# MentalBERT 모델 로드
# =============================================================================
print("⏳ MentalBERT 모델 로드 중...")
try:
    tokenizer = AutoTokenizer.from_pretrained(MENTAL_BERT_MODEL)
    mental_bert_model = AutoModel.from_pretrained(MENTAL_BERT_MODEL).to(DEVICE)
    mental_bert_model.eval()
    print("✅ MentalBERT 모델 로드 완료")
    print(f"   - 정신건강 도메인 특화 모델")
    print(f"   - Reddit mental health 데이터 사전학습\n")
except Exception as e:
    print(f"⚠️  MentalBERT 로드 실패, 일반 BERT 사용: {e}")
    from transformers import BertTokenizer, BertModel
    tokenizer = BertTokenizer.from_pretrained('bert-base-uncased')
    mental_bert_model = BertModel.from_pretrained('bert-base-uncased').to(DEVICE)
    mental_bert_model.eval()
    print("✅ 일반 BERT 모델 로드 완료\n")


# =============================================================================
# 유틸리티 함수
# =============================================================================
def remove_fillers_and_stopwords(text):
    """
    Filler words와 불용어 제거
    
    Args:
        text (str): 입력 텍스트
    
    Returns:
        str: 정제된 텍스트
    """
    if pd.isna(text) or text == '':
        return ''
    
    # 토큰화
    tokens = text.lower().split()
    
    # Filler words 및 불용어 제거
    cleaned_tokens = [
        token for token in tokens
        if token not in FILLER_WORDS and token not in STOPWORDS_TO_REMOVE
    ]
    
    return ' '.join(cleaned_tokens)


def clean_text(text):
    """
    텍스트 정제 (개선 버전)
    
    Args:
        text (str): 원본 텍스트
    
    Returns:
        str: 정제된 텍스트
    """
    if pd.isna(text) or text == '':
        return ''
    
    # 소문자 변환
    text = text.lower()
    
    # 특수 태그 제거 [PAUSE], [UNINTELLIGIBLE], [INAUDIBLE] 등
    text = re.sub(r'\[.*?\]', '', text)
    
    # 숫자만 있는 토큰 제거
    text = re.sub(r'\b\d+\b', '', text)
    
    # 특수문자 제거 (단, 느낌표와 물음표는 감정 표현이므로 공백으로 대체)
    text = re.sub(r'[!?]', ' ', text)  # 느낌표/물음표 → 공백
    text = re.sub(r'[^\w\s]', '', text)  # 나머지 특수문자 제거
    
    # Filler words 및 불용어 제거
    text = remove_fillers_and_stopwords(text)
    
    # 여러 공백을 하나로
    text = re.sub(r'\s+', ' ', text).strip()
    
    return text


def extract_mentalbert_embedding(text):
    """
    MentalBERT를 사용하여 텍스트 임베딩 추출
    
    Args:
        text (str): 입력 텍스트
    
    Returns:
        np.ndarray: (768,) MentalBERT embedding
    """
    if not text or text.strip() == '':
        # 빈 텍스트는 zero vector
        return np.zeros(768, dtype=np.float32)
    
    # Tokenize
    inputs = tokenizer(
        text,
        return_tensors='pt',
        padding=True,
        truncation=True,
        max_length=MAX_LENGTH
    ).to(DEVICE)
    
    # MentalBERT forward
    with torch.no_grad():
        outputs = mental_bert_model(**inputs)
    
    # [CLS] token embedding을 문장 대표 벡터로 사용
    cls_embedding = outputs.last_hidden_state[:, 0, :].cpu().numpy()
    
    return cls_embedding.squeeze(0).astype(np.float32)  # (768,)


def get_ttr(text):
    """
    Type-Token Ratio 계산
    
    Args:
        text (str): 입력 텍스트
    
    Returns:
        float: TTR 값
    """
    if not text or len(text.strip()) == 0:
        return 0.0
    tokens = text.lower().split()
    if len(tokens) == 0:
        return 0.0
    return len(set(tokens)) / len(tokens)


def calculate_linguistic_features(text):
    """
    추가 언어적 특징 계산
    
    Args:
        text (str): 입력 텍스트
    
    Returns:
        dict: 언어적 특징들
    """
    if not text or len(text.strip()) == 0:
        return {
            'word_count': 0,
            'avg_word_length': 0.0,
            'ttr': 0.0,
            'unique_words': 0
        }
    
    tokens = text.lower().split()
    unique_tokens = set(tokens)
    
    return {
        'word_count': len(tokens),
        'avg_word_length': np.mean([len(t) for t in tokens]) if tokens else 0.0,
        'ttr': len(unique_tokens) / len(tokens) if tokens else 0.0,
        'unique_words': len(unique_tokens)
    }


def normalize_question_type(q_type):
    """질문 유형 정규화 및 단순화"""
    q_type = q_type.lower().strip()
    q_type = ' '.join(q_type.split())
    q_type = q_type.replace('/', ' / ')
    q_type = ' '.join(q_type.split())
    
    if q_type in Q_TYPE_SIMPLIFICATION:
        return Q_TYPE_SIMPLIFICATION[q_type]
    return 'other'


# =============================================================================
# 메인 전처리 함수
# =============================================================================
def run_preprocessing_pipeline():
    """
    텍스트 데이터셋 전처리 (MentalBERT 기반)
    
    Returns:
        dict: 전처리된 데이터셋
    """
    
    # 1. 메타데이터 로드
    print("⏳ 메타데이터 로드 중...")
    meta = pd.read_csv(os.path.join(BASE_PATH, "metadataset.csv"))
    meta['Participant_ID'] = meta['Participant_ID'].astype(str)
    meta = meta[~meta['Participant_ID'].isin(EXCEPTION_NUMBER)].reset_index(drop=True)
    
    print(f"✅ 총 {len(meta)}명 참가자")
    print(f"   정상: {(meta['Binary'] == 0).sum()}명")
    print(f"   우울증: {(meta['Binary'] == 1).sum()}명\n")
    
    # 참가자별 데이터 저장
    dataset = {}
    
    print(f"\n{'='*70}")
    print(f"🚀 전처리 시작: {len(meta)}명")
    print(f"{'='*70}\n")
    
    stats = {
        'processed': 0,
        'failed': 0,
        'total_utterances': 0,
        'empty_after_cleaning': 0,
        'q_type_counts': defaultdict(int),
        'total_words_before': 0,
        'total_words_after': 0
    }
    
    for idx, row in tqdm(meta.iterrows(), total=len(meta), desc="참가자 처리"):
        pid = str(row['Participant_ID'])
        label = int(row['Binary'])
        
        # Transcript 파일 경로
        transcript_path = os.path.join(BASE_PATH, f"{pid}_P", f"{pid}_cleaned_transcript.csv")
        
        if not os.path.exists(transcript_path):
            stats['failed'] += 1
            continue
        
        # Transcript 로드
        try:
            df = pd.read_csv(transcript_path, sep='\t')
            if df.shape[1] < 2:
                df = pd.read_csv(transcript_path, sep=',')
        except Exception as e:
            stats['failed'] += 1
            continue
        
        # 컬럼명 정규화
        df.columns = [c.strip().lower().replace(' ', '_') for c in df.columns]
        
        # question_label 처리
        if 'question_label' not in df.columns:
            df['question_label'] = 'other'
        df['question_label'] = df['question_label'].fillna('other')
        df['question_label'] = df['question_label'].replace('', 'other')
        
        # 발화 추출
        processed_utterances = []
        current_q_type = None
        first_ellie_found = False
        
        for _, utt_row in df.iterrows():
            speaker = str(utt_row['speaker']).strip().lower()
            q_label = str(utt_row['question_label']).strip().lower()
            
            # Ellie 발화
            if 'ellie' in speaker:
                first_ellie_found = True
                
                # 질문 유형 업데이트
                q_label = normalize_question_type(q_label)
                if q_label != 'other':
                    current_q_type = q_label
                else:
                    current_q_type = 'other'
            
            # Participant 발화
            elif 'participant' in speaker:
                if not first_ellie_found:
                    continue
                
                if current_q_type is None:
                    continue
                
                text = str(utt_row['value'])
                
                # 정제 전 단어 수 (통계용)
                words_before = len(text.split())
                stats['total_words_before'] += words_before
                
                # 텍스트 정제 (개선된 버전)
                cleaned_text = clean_text(text)
                
                # 정제 후 빈 텍스트 체크
                if not cleaned_text or len(cleaned_text.split()) == 0:
                    stats['empty_after_cleaning'] += 1
                    continue
                
                # 정제 후 단어 수 (통계용)
                words_after = len(cleaned_text.split())
                stats['total_words_after'] += words_after
                
                # MentalBERT 임베딩 추출
                bert_embedding = extract_mentalbert_embedding(cleaned_text)
                
                # 언어적 특징 계산
                ling_features = calculate_linguistic_features(cleaned_text)
                
                # Q-type ID 변환
                q_type_id = Q_TYPE_MAPPING.get(current_q_type, Q_TYPE_MAPPING['other'])
                
                processed_utterances.append({
                    'bert': bert_embedding,  # (768,)
                    'q_type': current_q_type,
                    'q_type_id': q_type_id,
                    'ttr': ling_features['ttr'],
                    'word_count': ling_features['word_count'],
                    'avg_word_length': ling_features['avg_word_length'],
                    'unique_words': ling_features['unique_words'],
                    'text': cleaned_text  # 디버깅/분석용
                })
                
                stats['total_utterances'] += 1
                stats['q_type_counts'][current_q_type] += 1
        
        if not processed_utterances:
            stats['failed'] += 1
            continue
        
        # 참가자 데이터 저장
        dataset[pid] = {
            'label': label,
            'utterances': processed_utterances,
            'num_utterances': len(processed_utterances)
        }
        
        stats['processed'] += 1
    
    # 통계 출력
    print(f"\n{'='*70}")
    print(f"✅ 전처리 완료!")
    print(f"{'='*70}")
    print(f"처리 성공: {stats['processed']}명")
    print(f"처리 실패: {stats['failed']}명")
    print(f"총 발화: {stats['total_utterances']}개")
    print(f"정제 후 빈 발화: {stats['empty_after_cleaning']}개")
    
    # 단어 수 감소 통계
    reduction_rate = (1 - stats['total_words_after'] / stats['total_words_before']) * 100 if stats['total_words_before'] > 0 else 0
    print(f"\n단어 수 변화:")
    print(f"  정제 전: {stats['total_words_before']:,}개")
    print(f"  정제 후: {stats['total_words_after']:,}개")
    print(f"  감소율: {reduction_rate:.1f}%")
    
    print(f"\nQuestion Type 분포:")
    for q_type, count in sorted(stats['q_type_counts'].items(), key=lambda x: -x[1]):
        percentage = (count / stats['total_utterances']) * 100 if stats['total_utterances'] > 0 else 0
        print(f"  {q_type:15s}: {count:5d}개 ({percentage:5.1f}%)")
    
    return dataset


# =============================================================================
# 실행 및 저장
# =============================================================================
if __name__ == "__main__":
    
    # 전처리 실행
    dataset = run_preprocessing_pipeline()
    
    # 저장
    with open(OUTPUT_PATH, 'wb') as f:
        pickle.dump(dataset, f)
    
    print(f"\n💾 데이터 저장 완료: {OUTPUT_PATH}")
    
    # 파일 크기
    file_size_mb = os.path.getsize(OUTPUT_PATH) / (1024 * 1024)
    print(f"   파일 크기: {file_size_mb:.1f} MB")
    
    # 샘플 데이터 확인
    if len(dataset) > 0:
        sample_pid = list(dataset.keys())[0]
        sample = dataset[sample_pid]
        
        print(f"\n{'='*70}")
        print(f"📋 샘플 데이터 구조 (PID: {sample_pid})")
        print(f"{'='*70}")
        print(f"Label: {sample['label']}")
        print(f"Num Utterances: {sample['num_utterances']}")
        print(f"\n첫 번째 발화:")
        utt = sample['utterances'][0]
        print(f"  - Q-type: {utt['q_type']} (ID: {utt['q_type_id']})")
        print(f"  - TTR: {utt['ttr']:.3f}")
        print(f"  - Word Count: {utt['word_count']}")
        print(f"  - Avg Word Length: {utt['avg_word_length']:.2f}")
        print(f"  - Unique Words: {utt['unique_words']}")
        print(f"  - BERT Shape: {utt['bert'].shape}")
        print(f"  - Text: {utt['text'][:80]}...")
    
    print(f"\n{'='*70}")
    print(f"🎉 전처리 완료!")
    print(f"{'='*70}\n")
    
    print("개선사항:")
    print("  ✅ MentalBERT 사용 (정신건강 도메인 특화)")
    print("  ✅ Filler words 제거 (um, uh, hmm 등)")
    print("  ✅ NLTK 불용어 제거 (중요 단어 보존)")
    print("  ✅ 추가 언어적 특징 계산")
    
    print("\n다음 단계:")
    print("  1. 모델 학습 코드 실행")
    print("  2. 일반 BERT vs MentalBERT 성능 비교")
    print("  3. 텍스트 정제 효과 분석")

📝 텍스트 전처리 시작 (MentalBERT 기반)
Device: cuda
Model: mental/mental-bert-base-uncased
Filler words: 31개
Stopwords: 187개

⏳ MentalBERT 모델 로드 중...
⚠️  MentalBERT 로드 실패, 일반 BERT 사용: You are trying to access a gated repo.
Make sure to have access to it at https://huggingface.co/mental/mental-bert-base-uncased.
401 Client Error. (Request ID: Root=1-69317644-428ce8c86ad37a2f5d06e208;479b3fad-8cba-40fa-b3ae-cfaea6a80734)

Cannot access gated repo for url https://huggingface.co/mental/mental-bert-base-uncased/resolve/main/config.json.
Access to model mental/mental-bert-base-uncased is restricted. You must have access to it and be authenticated to access it. Please log in.
✅ 일반 BERT 모델 로드 완료

⏳ 메타데이터 로드 중...
✅ 총 186명 참가자
   정상: 129명
   우울증: 57명


🚀 전처리 시작: 186명



참가자 처리: 100%|██████████| 186/186 [04:47<00:00,  1.54s/it]



✅ 전처리 완료!
처리 성공: 186명
처리 실패: 0명
총 발화: 24910개
정제 후 빈 발화: 6106개

단어 수 변화:
  정제 전: 269,035개
  정제 후: 120,743개
  감소율: 55.1%

Question Type 분포:
  background     : 10204개 ( 41.0%)
  casual         :  7169개 ( 28.8%)
  emotional      :  4860개 ( 19.5%)
  clinical       :  2677개 ( 10.7%)

💾 데이터 저장 완료: D:\depression_dataset(DAIC-WOZ)\preprocessed_text_mentalbert_dataset.pkl
   파일 크기: 75.9 MB

📋 샘플 데이터 구조 (PID: 300)
Label: 0
Num Utterances: 77

첫 번째 발화:
  - Q-type: casual (ID: 0)
  - TTR: 1.000
  - Word Count: 1
  - Avg Word Length: 4.00
  - Unique Words: 1
  - BERT Shape: (768,)
  - Text: good...

🎉 전처리 완료!

개선사항:
  ✅ MentalBERT 사용 (정신건강 도메인 특화)
  ✅ Filler words 제거 (um, uh, hmm 등)
  ✅ NLTK 불용어 제거 (중요 단어 보존)
  ✅ 추가 언어적 특징 계산

다음 단계:
  1. 모델 학습 코드 실행
  2. 일반 BERT vs MentalBERT 성능 비교
  3. 텍스트 정제 효과 분석


In [25]:
"""
=============================================================================
텍스트 기반 우울증 감지 - 모델 학습
=============================================================================

특징: [BERT(768) + Q-type(32)] = 800차원
모델: Transformer (음성 모델과 동일 구조)
"""

import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
import numpy as np
import pandas as pd
import pickle
import os
import math
from sklearn.model_selection import train_test_split
from sklearn.metrics import f1_score, precision_score, recall_score, confusion_matrix, classification_report
from tqdm import tqdm
import matplotlib.pyplot as plt
import seaborn as sns

# =============================================================================
# 설정
# =============================================================================
BASE_PATH = r"D:\depression_dataset(DAIC-WOZ)"
PREPROCESSED_DATA_PATH = os.path.join(BASE_PATH, "preprocessed_text_dataset.pkl")

# 모델 하이퍼파라미터
BERT_DIM = 768
TTR_DIM = 1
Q_TYPE_VOCAB_SIZE = 5
Q_TYPE_EMBED_DIM = 32
INPUT_DIM = BERT_DIM + Q_TYPE_EMBED_DIM + TTR_DIM  # 801

D_MODEL = 256
NHEAD = 8
NUM_ENCODER_LAYERS = 3
DIM_FEEDFORWARD = 512
DROPOUT = 0.3

# 학습 하이퍼파라미터
BATCH_SIZE = 4  # 8 → 4 (더 작게)
NUM_EPOCHS = 50
LEARNING_RATE = 5e-5  # 1e-4 → 5e-5 (더 작게)
WEIGHT_DECAY = 1e-5  # 1e-4 → 1e-5 (더 작게)

# Loss 설정 (더 균형잡히게)
USE_FOCAL_LOSS = True
FOCAL_ALPHA = 0.65  # 0.55 → 0.65 (우울증에 더 집중)
FOCAL_GAMMA = 1.5   # 2.0 → 1.5 (덜 aggressive)
LABEL_SMOOTHING = 0.05  # 0.15 → 0.05 (덜 aggressive)

DYNAMIC_THRESHOLD = False  # 일단 끄기
MIN_RECALL_THRESHOLD = 0.75

EARLY_STOPPING_PATIENCE = 20  # 12 → 20 (더 기다리기)
GRADIENT_CLIP = 1.0

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

# Train/Val/Test split
TRAIN_RATIO = 0.7
VAL_RATIO = 0.15
TEST_RATIO = 0.15

print(f"{'='*70}")
print(f"📝 텍스트 기반 우울증 감지 모델 학습")
print(f"{'='*70}")
print(f"Input Dimension: {INPUT_DIM}")
print(f"Device: {DEVICE}")
print(f"{'='*70}\n")


# =============================================================================
# Loss Functions (음성 모델과 동일)
# =============================================================================
class FocalLoss(nn.Module):
    def __init__(self, alpha=0.75, gamma=2.0, label_smoothing=0.1):
        super(FocalLoss, self).__init__()
        self.alpha = alpha
        self.gamma = gamma
        self.label_smoothing = label_smoothing
    
    def forward(self, logits, targets):
        if self.label_smoothing > 0:
            targets = targets * (1 - self.label_smoothing) + self.label_smoothing / 2
        
        bce_loss = F.binary_cross_entropy_with_logits(logits, targets, reduction='none')
        pt = torch.exp(-bce_loss)
        focal_weight = (1 - pt) ** self.gamma
        alpha_weight = targets * self.alpha + (1 - targets) * (1 - self.alpha)
        focal_loss = alpha_weight * focal_weight * bce_loss
        
        return focal_loss.mean()


# =============================================================================
# Positional Encoding (음성 모델과 동일)
# =============================================================================
class PositionalEncoding(nn.Module):
    def __init__(self, d_model, max_len=500, dropout=0.1):
        super(PositionalEncoding, self).__init__()
        self.dropout = nn.Dropout(p=dropout)
        
        pe = torch.zeros(max_len, d_model)
        position = torch.arange(0, max_len, dtype=torch.float).unsqueeze(1)
        div_term = torch.exp(torch.arange(0, d_model, 2).float() * (-math.log(10000.0) / d_model))
        
        pe[:, 0::2] = torch.sin(position * div_term)
        pe[:, 1::2] = torch.cos(position * div_term)
        pe = pe.unsqueeze(0)
        
        self.register_buffer('pe', pe)
    
    def forward(self, x):
        x = x + self.pe[:, :x.size(1), :]
        return self.dropout(x)


# =============================================================================
# Transformer 모델 (음성 모델과 거의 동일, input_dim만 변경)
# =============================================================================
class TransformerTextDepressionModel(nn.Module):
    def __init__(
        self,
        input_dim=INPUT_DIM,
        d_model=D_MODEL,
        nhead=NHEAD,
        num_encoder_layers=NUM_ENCODER_LAYERS,
        dim_feedforward=DIM_FEEDFORWARD,
        dropout=DROPOUT,
        q_type_vocab_size=Q_TYPE_VOCAB_SIZE,
        q_type_embed_dim=Q_TYPE_EMBED_DIM
    ):
        super(TransformerTextDepressionModel, self).__init__()
        
        self.d_model = d_model
        
        # Question Type 임베딩
        self.q_type_embedding = nn.Embedding(q_type_vocab_size, q_type_embed_dim)
        
        # Input projection
        self.input_projection = nn.Linear(input_dim, d_model)
        
        # Positional Encoding
        self.pos_encoder = PositionalEncoding(d_model, dropout=dropout)
        
        # Transformer Encoder
        encoder_layer = nn.TransformerEncoderLayer(
            d_model=d_model,
            nhead=nhead,
            dim_feedforward=dim_feedforward,
            dropout=dropout,
            activation='gelu',
            batch_first=True
        )
        
        self.transformer_encoder = nn.TransformerEncoder(
            encoder_layer,
            num_layers=num_encoder_layers
        )
        
        # [CLS] token
        self.cls_token = nn.Parameter(torch.randn(1, 1, d_model))
        
        # Classifier
        self.classifier = nn.Sequential(
            nn.LayerNorm(d_model),
            nn.Dropout(dropout),
            nn.Linear(d_model, d_model // 2),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(d_model // 2, 1)
        )
        
        self.dropout = nn.Dropout(dropout)
    
    def forward(self, batch_bert, batch_ttrs, batch_q_type_ids, num_utterances_list):
        """
        Args:
            batch_bert: [total_utterances, 768]
            batch_ttrs: [total_utterances]
            batch_q_type_ids: [total_utterances]
            num_utterances_list: [batch_size]
        """
        batch_size = len(num_utterances_list)
        device = batch_bert.device
        
        # Feature 결합
        q_type_embs = self.q_type_embedding(batch_q_type_ids)  # [total_utts, 32]
        ttrs_expanded = batch_ttrs.unsqueeze(1)  # [total_utts, 1]
        
        combined_features = torch.cat([
            batch_bert,      # [total_utts, 768]
            q_type_embs,     # [total_utts, 32]
            ttrs_expanded    # [total_utts, 1]
        ], dim=1)  # [total_utts, 801]
        
        # 참가자별로 재구성
        participant_sequences = []
        start_idx = 0
        
        for num_utts in num_utterances_list:
            end_idx = start_idx + num_utts
            participant_sequences.append(combined_features[start_idx:end_idx])
            start_idx = end_idx
        
        # 패딩 및 마스크
        max_num_utterances = max(num_utterances_list)
        padded_sequences = []
        attention_masks = []
        
        for seq, num_utts in zip(participant_sequences, num_utterances_list):
            if num_utts < max_num_utterances:
                padding = torch.zeros(
                    max_num_utterances - num_utts,
                    seq.size(1),
                    device=device
                )
                seq = torch.cat([seq, padding], dim=0)
            
            padded_sequences.append(seq)
            
            mask = torch.zeros(max_num_utterances, dtype=torch.bool, device=device)
            mask[num_utts:] = True
            attention_masks.append(mask)
        
        padded_sequences = torch.stack(padded_sequences)  # [B, max_utts, 800]
        attention_masks = torch.stack(attention_masks)    # [B, max_utts]
        
        # Input Projection
        x = self.input_projection(padded_sequences)  # [B, max_utts, d_model]
        
        # [CLS] token 추가
        cls_tokens = self.cls_token.expand(batch_size, -1, -1)
        x = torch.cat([cls_tokens, x], dim=1)  # [B, max_utts+1, d_model]
        
        cls_mask = torch.zeros(batch_size, 1, dtype=torch.bool, device=device)
        extended_mask = torch.cat([cls_mask, attention_masks], dim=1)
        
        # Positional Encoding
        x = self.pos_encoder(x)
        
        # Transformer Encoder
        encoded = self.transformer_encoder(
            x,
            src_key_padding_mask=extended_mask
        )
        
        # [CLS] token 추출
        cls_output = encoded[:, 0, :]
        
        # Classification
        logits = self.classifier(cls_output)
        
        # Attention weights (해석용)
        utterance_features = encoded[:, 1:, :]
        attention_weights = torch.norm(utterance_features, dim=2)
        attention_weights = attention_weights.masked_fill(attention_masks, 0)
        attention_weights = F.softmax(attention_weights + (attention_masks.float() * -1e9), dim=1)
        
        return logits, attention_weights


# =============================================================================
# Dataset & Collate (음성과 유사, 특징만 변경)
# =============================================================================
class TextUtteranceDataset(Dataset):
    def __init__(self, participant_data):
        self.data = []
        for pid, info in participant_data.items():
            self.data.append({
                'pid': pid,
                'label': info['label'],
                'utterances': info['utterances'],
                'num_utterances': info['num_utterances']
            })
    
    def __len__(self):
        return len(self.data)
    
    def __getitem__(self, idx):
        return self.data[idx]


def collate_fn(batch):
    batch_labels = []
    all_utterances = []
    num_utterances_list = []
    batch_pids = []
    
    for item in batch:
        batch_labels.append(item['label'])
        all_utterances.extend(item['utterances'])
        num_utterances_list.append(item['num_utterances'])
        batch_pids.append(item['pid'])
    
    # 특징 추출
    batch_bert = []
    batch_ttrs = []
    batch_q_type_ids = []
    
    for utt in all_utterances:
        batch_bert.append(utt['bert'])
        batch_ttrs.append(utt['ttr'])
        batch_q_type_ids.append(utt['q_type_id'])
    
    # Tensor 변환
    batch_bert = torch.FloatTensor(np.array(batch_bert))  # [total_utterances, 768]
    batch_ttrs = torch.FloatTensor(batch_ttrs)  # [total_utterances]
    batch_q_type_ids = torch.LongTensor(batch_q_type_ids)  # [total_utterances]
    batch_labels = torch.FloatTensor(batch_labels).unsqueeze(1)  # [batch_size, 1]
    
    return {
        'batch_bert': batch_bert,
        'batch_ttrs': batch_ttrs,
        'batch_q_type_ids': batch_q_type_ids,
        'batch_labels': batch_labels,
        'num_utterances_list': num_utterances_list,
        'batch_pids': batch_pids
    }


# =============================================================================
# 데이터 로드 및 분할
# =============================================================================
def load_and_split_data():
    print(f"{'='*70}")
    print(f"📂 데이터 로드 중...")
    print(f"{'='*70}")
    
    with open(PREPROCESSED_DATA_PATH, 'rb') as f:
        dataset = pickle.load(f)
    
    # 메타데이터 로드하여 Group 정보 사용
    meta_df = pd.read_csv(os.path.join(BASE_PATH, "metadataset.csv"))
    meta_df['Participant_ID'] = meta_df['Participant_ID'].astype(str)
    
    # Group 열을 기준으로 분할
    train_pids = meta_df[meta_df['Group'] == 'Train']['Participant_ID'].tolist()
    val_pids = meta_df[meta_df['Group'] == 'Validation']['Participant_ID'].tolist()
    test_pids = meta_df[meta_df['Group'] == 'Test']['Participant_ID'].tolist()
    
    # dataset에 있는 PID만 사용 (전처리된 데이터 기준)
    train_pids = [pid for pid in train_pids if pid in dataset]
    val_pids = [pid for pid in val_pids if pid in dataset]
    test_pids = [pid for pid in test_pids if pid in dataset]
    
    # 라벨 추출
    train_labels = [dataset[pid]['label'] for pid in train_pids]
    val_labels = [dataset[pid]['label'] for pid in val_pids]
    test_labels = [dataset[pid]['label'] for pid in test_pids]
    
    print(f"총 참가자 수: {len(dataset)}")
    print(f"  - 정상 (0): {sum(1 for d in dataset.values() if d['label'] == 0)}")
    print(f"  - 우울증 (1): {sum(1 for d in dataset.values() if d['label'] == 1)}")
    
    print(f"\n메타데이터 기준 분할:")
    print(f"  - Train: {len(train_pids)} (0: {train_labels.count(0)}, 1: {train_labels.count(1)})")
    print(f"  - Val:   {len(val_pids)} (0: {val_labels.count(0)}, 1: {val_labels.count(1)})")
    print(f"  - Test:  {len(test_pids)} (0: {test_labels.count(0)}, 1: {test_labels.count(1)})")
    
    # 데이터셋 생성
    train_data = {pid: dataset[pid] for pid in train_pids}
    val_data = {pid: dataset[pid] for pid in val_pids}
    test_data = {pid: dataset[pid] for pid in test_pids}
    
    train_dataset = TextUtteranceDataset(train_data)
    val_dataset = TextUtteranceDataset(val_data)
    test_dataset = TextUtteranceDataset(test_data)
    
    # DataLoader 생성
    train_loader = DataLoader(
        train_dataset, batch_size=BATCH_SIZE, shuffle=True,
        collate_fn=collate_fn, num_workers=0, pin_memory=True if torch.cuda.is_available() else False
    )
    val_loader = DataLoader(
        val_dataset, batch_size=BATCH_SIZE, shuffle=False,
        collate_fn=collate_fn, num_workers=0, pin_memory=True if torch.cuda.is_available() else False
    )
    test_loader = DataLoader(
        test_dataset, batch_size=BATCH_SIZE, shuffle=False,
        collate_fn=collate_fn, num_workers=0, pin_memory=True if torch.cuda.is_available() else False
    )
    
    return train_loader, val_loader, test_loader


# =============================================================================
# 평가 함수 (음성 모델과 동일)
# =============================================================================
def evaluate(model, dataloader, criterion, threshold=0.5, return_all_thresholds=False):
    model.eval()
    
    all_labels = []
    all_probs = []
    all_preds = []
    total_loss = 0.0
    
    with torch.no_grad():
        for batch in dataloader:
            batch_bert = batch['batch_bert'].to(DEVICE)
            batch_ttrs = batch['batch_ttrs'].to(DEVICE)
            batch_q_type_ids = batch['batch_q_type_ids'].to(DEVICE)
            batch_labels = batch['batch_labels'].to(DEVICE)
            
            logits, _ = model(
                batch_bert,
                batch_ttrs,
                batch_q_type_ids,
                batch['num_utterances_list']
            )
            
            loss = criterion(logits, batch_labels)
            total_loss += loss.item()
            
            probs = torch.sigmoid(logits)
            preds = (probs > threshold).float()
            
            all_labels.extend(batch_labels.cpu().numpy().flatten())
            all_probs.extend(probs.cpu().numpy().flatten())
            all_preds.extend(preds.cpu().numpy().flatten())
    
    avg_loss = total_loss / len(dataloader)
    f1 = f1_score(all_labels, all_preds)
    precision = precision_score(all_labels, all_preds, zero_division=0)
    recall = recall_score(all_labels, all_preds, zero_division=0)
    
    return {
        'loss': avg_loss,
        'f1': f1,
        'precision': precision,
        'recall': recall,
        'labels': all_labels,
        'probs': all_probs,
        'preds': all_preds
    }


def find_optimal_threshold(labels, probs, min_recall=0.75):
    best_f1 = 0.0
    best_threshold = 0.5
    best_metrics = {}
    
    for thresh in np.arange(0.1, 0.7, 0.02):
        preds = (np.array(probs) > thresh).astype(int)
        f1 = f1_score(labels, preds)
        recall = recall_score(labels, preds, zero_division=0)
        precision = precision_score(labels, preds, zero_division=0)
        
        if recall < min_recall:
            continue
        
        if f1 > best_f1:
            best_f1 = f1
            best_threshold = thresh
            best_metrics = {'f1': f1, 'precision': precision, 'recall': recall, 'threshold': thresh}
    
    return best_threshold, best_metrics


# =============================================================================
# Training Loop (음성과 동일, forward만 수정)
# =============================================================================
def train_model(model, train_loader, val_loader, num_epochs=NUM_EPOCHS):
    criterion = FocalLoss(alpha=FOCAL_ALPHA, gamma=FOCAL_GAMMA, label_smoothing=LABEL_SMOOTHING)
    
    optimizer = optim.AdamW(model.parameters(), lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY)
    
    warmup_epochs = 5
    warmup_scheduler = optim.lr_scheduler.LinearLR(optimizer, start_factor=0.1, end_factor=1.0, total_iters=warmup_epochs)
    cosine_scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=num_epochs - warmup_epochs, eta_min=1e-6)
    scheduler = optim.lr_scheduler.SequentialLR(optimizer, schedulers=[warmup_scheduler, cosine_scheduler], milestones=[warmup_epochs])
    
    best_val_f1 = 0.0
    best_val_precision = 0.0
    best_val_recall = 0.0
    patience_counter = 0
    
    history = {'train_loss': [], 'val_loss': [], 'val_f1': [], 'val_precision': [], 'val_recall': []}
    
    print(f"\n{'='*70}")
    print(f"🚀 학습 시작")
    print(f"{'='*70}\n")
    
    for epoch in range(num_epochs):
        model.train()
        train_loss = 0.0
        
        pbar = tqdm(train_loader, desc=f"Epoch {epoch+1}/{num_epochs}")
        for batch in pbar:
            batch_bert = batch['batch_bert'].to(DEVICE)
            batch_ttrs = batch['batch_ttrs'].to(DEVICE)
            batch_q_type_ids = batch['batch_q_type_ids'].to(DEVICE)
            batch_labels = batch['batch_labels'].to(DEVICE)
            
            optimizer.zero_grad()
            logits, _ = model(batch_bert, batch_ttrs, batch_q_type_ids, batch['num_utterances_list'])
            
            loss = criterion(logits, batch_labels)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), GRADIENT_CLIP)
            optimizer.step()
            
            train_loss += loss.item()
            pbar.set_postfix({'loss': loss.item()})
        
        avg_train_loss = train_loss / len(train_loader)
        
        val_results = evaluate(model, val_loader, criterion, threshold=0.5)
        
        history['train_loss'].append(avg_train_loss)
        history['val_loss'].append(val_results['loss'])
        history['val_f1'].append(val_results['f1'])
        history['val_precision'].append(val_results['precision'])
        history['val_recall'].append(val_results['recall'])
        
        scheduler.step()
        
        print(f"\nEpoch {epoch+1}/{num_epochs}")
        print(f"  Train Loss: {avg_train_loss:.4f}")
        print(f"  Val Loss:   {val_results['loss']:.4f}")
        print(f"  Val F1:     {val_results['f1']:.4f}")
        print(f"  Val Prec:   {val_results['precision']:.4f}")
        print(f"  Val Recall: {val_results['recall']:.4f}")
        
        # 🔍 디버깅: 예측 분포 확인
        val_preds_sum = sum(val_results['preds'])
        val_labels_sum = sum(val_results['labels'])
        val_probs_array = np.array(val_results['probs'])
        print(f"  [DEBUG] Val 예측 분포: {val_preds_sum}/{len(val_results['preds'])} 우울증 예측")
        print(f"  [DEBUG] Val 실제 분포: {val_labels_sum}/{len(val_results['labels'])} 우울증 실제")
        print(f"  [DEBUG] Val 예측 확률: min={val_probs_array.min():.3f}, max={val_probs_array.max():.3f}, mean={val_probs_array.mean():.3f}")
        
        # 조건: F1 개선 또는 첫 epoch
        improved = val_results['f1'] > best_val_f1
        
        if improved or epoch == 0:  # 첫 epoch는 무조건 저장
            if improved:
                best_val_f1 = val_results['f1']
                best_val_precision = val_results['precision']
                best_val_recall = val_results['recall']
            
            patience_counter = 0
            
            torch.save({
                'epoch': epoch,
                'model_state_dict': model.state_dict(),
                'optimizer_state_dict': optimizer.state_dict(),
                'val_f1': val_results['f1'],
                'val_precision': val_results['precision'],
                'val_recall': val_results['recall'],
                'history': history
            }, os.path.join(BASE_PATH, 'best_text_model.pt'))
            
            if epoch == 0:
                print(f"  💾 초기 모델 저장 (F1: {val_results['f1']:.4f})")
            else:
                print(f"  ✅ Best model saved! (F1: {best_val_f1:.4f})")
        else:
            patience_counter += 1
            print(f"  ⏳ No improvement ({patience_counter}/{EARLY_STOPPING_PATIENCE})")
            
            if patience_counter >= EARLY_STOPPING_PATIENCE:
                print(f"\n⚠️  Early stopping!")
                break
        
        print("-" * 70)
    
    return history


# =============================================================================
# 메인 실행
# =============================================================================
if __name__ == "__main__":
    
    # 데이터 로드
    train_loader, val_loader, test_loader = load_and_split_data()
    
    # 모델 초기화
    model = TransformerTextDepressionModel(
        input_dim=INPUT_DIM,
        d_model=D_MODEL,
        nhead=NHEAD,
        num_encoder_layers=NUM_ENCODER_LAYERS,
        dim_feedforward=DIM_FEEDFORWARD,
        dropout=DROPOUT,
        q_type_vocab_size=Q_TYPE_VOCAB_SIZE,
        q_type_embed_dim=Q_TYPE_EMBED_DIM
    ).to(DEVICE)
    
    total_params = sum(p.numel() for p in model.parameters())
    print(f"\n{'='*70}")
    print(f"🤖 모델 초기화 완료")
    print(f"{'='*70}")
    print(f"총 파라미터 수: {total_params:,}")
    print(f"Device: {DEVICE}")
    print(f"{'='*70}\n")
    
    # 학습
    history = train_model(model, train_loader, val_loader)
    
    # Test 평가
    model_path = os.path.join(BASE_PATH, 'best_text_model.pt')
    
    if os.path.exists(model_path):
        print(f"\n{'='*70}")
        print(f"📊 Test Set 평가")
        print(f"{'='*70}\n")
        
        checkpoint = torch.load(model_path)
        model.load_state_dict(checkpoint['model_state_dict'])
        
        # 먼저 전체 확률 추출
        test_results_full = evaluate(model, test_loader, nn.BCEWithLogitsLoss(), threshold=0.5)
        all_probs = np.array(test_results_full['probs'])
        all_labels = np.array(test_results_full['labels'])
        
        # Threshold 최적화
        print("🎯 Threshold 최적화:")
        print("Threshold | F1    | Prec  | Rec   | Spec")
        print("-" * 50)
        
        best_f1 = 0.0
        best_threshold = 0.5
        best_metrics = {}
        
        for thresh in np.arange(0.2, 0.8, 0.05):
            preds = (all_probs > thresh).astype(int)
            
            f1 = f1_score(all_labels, preds)
            precision = precision_score(all_labels, preds, zero_division=0)
            recall = recall_score(all_labels, preds, zero_division=0)
            
            tn = sum((l == 0 and p == 0) for l, p in zip(all_labels, preds))
            fp = sum((l == 0 and p == 1) for l, p in zip(all_labels, preds))
            specificity = tn / (tn + fp) if (tn + fp) > 0 else 0
            
            if f1 > best_f1:
                best_f1 = f1
                best_threshold = thresh
                best_metrics = {
                    'f1': f1,
                    'precision': precision,
                    'recall': recall,
                    'specificity': specificity
                }
            
            marker = "⭐" if abs(thresh - best_threshold) < 0.01 else ""
            print(f"{thresh:.2f}      | {f1:.3f} | {precision:.3f} | {recall:.3f} | {specificity:.3f} {marker}")
        
        print(f"\n{'='*70}")
        print(f"💡 최적 Threshold: {best_threshold:.2f}")
        print(f"{'='*70}")
        print(f"최종 Test 성능 (Threshold={best_threshold:.2f}):")
        print(f"  F1:          {best_metrics['f1']:.4f}")
        print(f"  Precision:   {best_metrics['precision']:.4f}")
        print(f"  Recall:      {best_metrics['recall']:.4f}")
        print(f"  Specificity: {best_metrics['specificity']:.4f}")
        
        print(f"\n✅ 학습 및 평가 완료!")
        print(f"\n{'='*70}")
        print(f"📊 음성 vs 텍스트 비교")
        print(f"{'='*70}")
        print(f"음성 (Threshold=0.68):")
        print(f"  F1: 0.700, Precision: 0.583, Recall: 0.875")
        print(f"\n텍스트 (Threshold={best_threshold:.2f}):")
        print(f"  F1: {best_metrics['f1']:.3f}, Precision: {best_metrics['precision']:.3f}, Recall: {best_metrics['recall']:.3f}")
    else:
        print(f"\n⚠️ 모델 파일을 찾을 수 없습니다: {model_path}")
        print(f"   학습 중 early stopping이 발동하지 않았거나 에러가 발생했을 수 있습니다.")
        print(f"\n✅ 학습 완료!")

📝 텍스트 기반 우울증 감지 모델 학습
Input Dimension: 801
Device: cuda

📂 데이터 로드 중...
총 참가자 수: 186
  - 정상 (0): 129
  - 우울증 (1): 57

메타데이터 기준 분할:
  - Train: 107 (0: 76, 1: 31)
  - Val:   33 (0: 21, 1: 12)
  - Test:  46 (0: 32, 1: 14)

🤖 모델 초기화 완료
총 파라미터 수: 1,820,577
Device: cuda


🚀 학습 시작



Epoch 1/50: 100%|██████████| 27/27 [00:00<00:00, 33.04it/s, loss=0.062] 



Epoch 1/50
  Train Loss: 0.1118
  Val Loss:   0.1125
  Val F1:     0.0000
  Val Prec:   0.0000
  Val Recall: 0.0000
  [DEBUG] Val 예측 분포: 0.0/33 우울증 예측
  [DEBUG] Val 실제 분포: 12.0/33 우울증 실제
  [DEBUG] Val 예측 확률: min=0.449, max=0.466, mean=0.454
  💾 초기 모델 저장 (F1: 0.0000)
----------------------------------------------------------------------


Epoch 2/50: 100%|██████████| 27/27 [00:00<00:00, 48.66it/s, loss=0.0966]



Epoch 2/50
  Train Loss: 0.1141
  Val Loss:   0.1159
  Val F1:     0.0000
  Val Prec:   0.0000
  Val Recall: 0.0000
  [DEBUG] Val 예측 분포: 0.0/33 우울증 예측
  [DEBUG] Val 실제 분포: 12.0/33 우울증 실제
  [DEBUG] Val 예측 확률: min=0.421, max=0.437, mean=0.426
  ⏳ No improvement (1/20)
----------------------------------------------------------------------


...


Epoch 19/50
  Train Loss: 0.1058
  Val Loss:   0.1103
  Val F1:     0.0000
  Val Prec:   0.0000
  Val Recall: 0.0000
  [DEBUG] Val 예측 분포: 1.0/33 우울증 예측
  [DEBUG] Val 실제 분포: 12.0/33 우울증 실제
  [DEBUG] Val 예측 확률: min=0.451, max=0.513, mean=0.472
  ⏳ No improvement (18/20)
----------------------------------------------------------------------


Epoch 20/50: 100%|██████████| 27/27 [00:00<00:00, 48.63it/s, loss=0.184] 



Epoch 20/50
  Train Loss: 0.1078
  Val Loss:   0.1170
  Val F1:     0.0000
  Val Prec:   0.0000
  Val Recall: 0.0000
  [DEBUG] Val 예측 분포: 0.0/33 우울증 예측
  [DEBUG] Val 실제 분포: 12.0/33 우울증 실제
  [DEBUG] Val 예측 확률: min=0.392, max=0.460, mean=0.415
  ⏳ No improvement (19/20)
----------------------------------------------------------------------


Epoch 21/50: 100%|██████████| 27/27 [00:00<00:00, 50.72it/s, loss=0.124] 



Epoch 21/50
  Train Loss: 0.1068
  Val Loss:   0.1104
  Val F1:     0.0000
  Val Prec:   0.0000
  Val Recall: 0.0000
  [DEBUG] Val 예측 분포: 1.0/33 우울증 예측
  [DEBUG] Val 실제 분포: 12.0/33 우울증 실제
  [DEBUG] Val 예측 확률: min=0.432, max=0.513, mean=0.460
  ⏳ No improvement (20/20)

⚠️  Early stopping!

📊 Test Set 평가

🎯 Threshold 최적화:
Threshold | F1    | Prec  | Rec   | Spec
--------------------------------------------------
0.20      | 0.467 | 0.304 | 1.000 | 0.000 ⭐
0.25      | 0.467 | 0.304 | 1.000 | 0.000 
0.30      | 0.467 | 0.304 | 1.000 | 0.000 
0.35      | 0.467 | 0.304 | 1.000 | 0.000 
0.40      | 0.467 | 0.304 | 1.000 | 0.000 
0.45      | 0.483 | 0.318 | 1.000 | 0.062 ⭐
0.50      | 0.000 | 0.000 | 0.000 | 1.000 
0.55      | 0.000 | 0.000 | 0.000 | 1.000 
0.60      | 0.000 | 0.000 | 0.000 | 1.000 
0.65      | 0.000 | 0.000 | 0.000 | 1.000 
0.70      | 0.000 | 0.000 | 0.000 | 1.000 
0.75      | 0.000 | 0.000 | 0.000 | 1.000 
0.80      | 0.000 | 0.000 | 0.000 | 1.000 

💡 최적 Threshold: 0.45
최

텍스트 모델은 되려 붕괴현상이 있음 focal loss를 끄기로 결정

In [26]:
"""
=============================================================================
텍스트 기반 우울증 감지 - 모델 학습
=============================================================================

특징: [BERT(768) + Q-type(32)] = 800차원
모델: Transformer (음성 모델과 동일 구조)
"""

import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
import numpy as np
import pandas as pd
import pickle
import os
import math
from sklearn.model_selection import train_test_split
from sklearn.metrics import f1_score, precision_score, recall_score, confusion_matrix, classification_report
from tqdm import tqdm
import matplotlib.pyplot as plt
import seaborn as sns

# =============================================================================
# 설정
# =============================================================================
BASE_PATH = r"D:\depression_dataset(DAIC-WOZ)"
PREPROCESSED_DATA_PATH = os.path.join(BASE_PATH, "preprocessed_text_dataset.pkl")

# 모델 하이퍼파라미터
BERT_DIM = 768
TTR_DIM = 1
Q_TYPE_VOCAB_SIZE = 5
Q_TYPE_EMBED_DIM = 32
INPUT_DIM = BERT_DIM + Q_TYPE_EMBED_DIM + TTR_DIM  # 801

D_MODEL = 256
NHEAD = 8
NUM_ENCODER_LAYERS = 3
DIM_FEEDFORWARD = 512
DROPOUT = 0.3

POS_WEIGHT = 1.0
LEARNING_RATE = 1e-5  # 매우 낮게
NUM_EPOCHS = 100
BATCH_SIZE = 2  # 더 작게
# 학습 하이퍼파라미터
#BATCH_SIZE = 4  # 8 → 4 (더 작게)
#NUM_EPOCHS = 50
#LEARNING_RATE = 5e-5  # 1e-4 → 5e-5 (더 작게)
WEIGHT_DECAY = 1e-5  # 1e-4 → 1e-5 (더 작게)

# Loss 설정 (Focal Loss OFF - 단순 BCE 사용)
USE_FOCAL_LOSS = False  # True → False
FOCAL_ALPHA = 0.65  # 사용 안 함
FOCAL_GAMMA = 1.5   # 사용 안 함
LABEL_SMOOTHING = 0.05  # 사용 안 함

# Balanced BCE pos_weight 계산
# Train: 76 정상, 31 우울증 → pos_weight = 76/31 ≈ 2.45
POS_WEIGHT = 2.45  # 클래스 불균형 보정

DYNAMIC_THRESHOLD = False
MIN_RECALL_THRESHOLD = 0.75

EARLY_STOPPING_PATIENCE = 20
GRADIENT_CLIP = 1.0

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

# Train/Val/Test split
TRAIN_RATIO = 0.7
VAL_RATIO = 0.15
TEST_RATIO = 0.15

print(f"{'='*70}")
print(f"📝 텍스트 기반 우울증 감지 모델 학습")
print(f"{'='*70}")
print(f"Input Dimension: {INPUT_DIM}")
print(f"Device: {DEVICE}")
print(f"{'='*70}\n")


# =============================================================================
# Loss Functions (음성 모델과 동일)
# =============================================================================
class FocalLoss(nn.Module):
    def __init__(self, alpha=0.75, gamma=2.0, label_smoothing=0.1):
        super(FocalLoss, self).__init__()
        self.alpha = alpha
        self.gamma = gamma
        self.label_smoothing = label_smoothing
    
    def forward(self, logits, targets):
        if self.label_smoothing > 0:
            targets = targets * (1 - self.label_smoothing) + self.label_smoothing / 2
        
        bce_loss = F.binary_cross_entropy_with_logits(logits, targets, reduction='none')
        pt = torch.exp(-bce_loss)
        focal_weight = (1 - pt) ** self.gamma
        alpha_weight = targets * self.alpha + (1 - targets) * (1 - self.alpha)
        focal_loss = alpha_weight * focal_weight * bce_loss
        
        return focal_loss.mean()


# =============================================================================
# Positional Encoding (음성 모델과 동일)
# =============================================================================
class PositionalEncoding(nn.Module):
    def __init__(self, d_model, max_len=500, dropout=0.1):
        super(PositionalEncoding, self).__init__()
        self.dropout = nn.Dropout(p=dropout)
        
        pe = torch.zeros(max_len, d_model)
        position = torch.arange(0, max_len, dtype=torch.float).unsqueeze(1)
        div_term = torch.exp(torch.arange(0, d_model, 2).float() * (-math.log(10000.0) / d_model))
        
        pe[:, 0::2] = torch.sin(position * div_term)
        pe[:, 1::2] = torch.cos(position * div_term)
        pe = pe.unsqueeze(0)
        
        self.register_buffer('pe', pe)
    
    def forward(self, x):
        x = x + self.pe[:, :x.size(1), :]
        return self.dropout(x)


# =============================================================================
# Transformer 모델 (음성 모델과 거의 동일, input_dim만 변경)
# =============================================================================
class TransformerTextDepressionModel(nn.Module):
    def __init__(
        self,
        input_dim=INPUT_DIM,
        d_model=D_MODEL,
        nhead=NHEAD,
        num_encoder_layers=NUM_ENCODER_LAYERS,
        dim_feedforward=DIM_FEEDFORWARD,
        dropout=DROPOUT,
        q_type_vocab_size=Q_TYPE_VOCAB_SIZE,
        q_type_embed_dim=Q_TYPE_EMBED_DIM
    ):
        super(TransformerTextDepressionModel, self).__init__()
        
        self.d_model = d_model
        
        # Question Type 임베딩
        self.q_type_embedding = nn.Embedding(q_type_vocab_size, q_type_embed_dim)
        
        # Input projection
        self.input_projection = nn.Linear(input_dim, d_model)
        
        # Positional Encoding
        self.pos_encoder = PositionalEncoding(d_model, dropout=dropout)
        
        # Transformer Encoder
        encoder_layer = nn.TransformerEncoderLayer(
            d_model=d_model,
            nhead=nhead,
            dim_feedforward=dim_feedforward,
            dropout=dropout,
            activation='gelu',
            batch_first=True
        )
        
        self.transformer_encoder = nn.TransformerEncoder(
            encoder_layer,
            num_layers=num_encoder_layers
        )
        
        # [CLS] token
        self.cls_token = nn.Parameter(torch.randn(1, 1, d_model))
        
        # Classifier
        self.classifier = nn.Sequential(
            nn.LayerNorm(d_model),
            nn.Dropout(dropout),
            nn.Linear(d_model, d_model // 2),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(d_model // 2, 1)
        )
        
        self.dropout = nn.Dropout(dropout)
    
    def forward(self, batch_bert, batch_ttrs, batch_q_type_ids, num_utterances_list):
        """
        Args:
            batch_bert: [total_utterances, 768]
            batch_ttrs: [total_utterances]
            batch_q_type_ids: [total_utterances]
            num_utterances_list: [batch_size]
        """
        batch_size = len(num_utterances_list)
        device = batch_bert.device
        
        # Feature 결합
        q_type_embs = self.q_type_embedding(batch_q_type_ids)  # [total_utts, 32]
        ttrs_expanded = batch_ttrs.unsqueeze(1)  # [total_utts, 1]
        
        combined_features = torch.cat([
            batch_bert,      # [total_utts, 768]
            q_type_embs,     # [total_utts, 32]
            ttrs_expanded    # [total_utts, 1]
        ], dim=1)  # [total_utts, 801]
        
        # 참가자별로 재구성
        participant_sequences = []
        start_idx = 0
        
        for num_utts in num_utterances_list:
            end_idx = start_idx + num_utts
            participant_sequences.append(combined_features[start_idx:end_idx])
            start_idx = end_idx
        
        # 패딩 및 마스크
        max_num_utterances = max(num_utterances_list)
        padded_sequences = []
        attention_masks = []
        
        for seq, num_utts in zip(participant_sequences, num_utterances_list):
            if num_utts < max_num_utterances:
                padding = torch.zeros(
                    max_num_utterances - num_utts,
                    seq.size(1),
                    device=device
                )
                seq = torch.cat([seq, padding], dim=0)
            
            padded_sequences.append(seq)
            
            mask = torch.zeros(max_num_utterances, dtype=torch.bool, device=device)
            mask[num_utts:] = True
            attention_masks.append(mask)
        
        padded_sequences = torch.stack(padded_sequences)  # [B, max_utts, 800]
        attention_masks = torch.stack(attention_masks)    # [B, max_utts]
        
        # Input Projection
        x = self.input_projection(padded_sequences)  # [B, max_utts, d_model]
        
        # [CLS] token 추가
        cls_tokens = self.cls_token.expand(batch_size, -1, -1)
        x = torch.cat([cls_tokens, x], dim=1)  # [B, max_utts+1, d_model]
        
        cls_mask = torch.zeros(batch_size, 1, dtype=torch.bool, device=device)
        extended_mask = torch.cat([cls_mask, attention_masks], dim=1)
        
        # Positional Encoding
        x = self.pos_encoder(x)
        
        # Transformer Encoder
        encoded = self.transformer_encoder(
            x,
            src_key_padding_mask=extended_mask
        )
        
        # [CLS] token 추출
        cls_output = encoded[:, 0, :]
        
        # Classification
        logits = self.classifier(cls_output)
        
        # Attention weights (해석용)
        utterance_features = encoded[:, 1:, :]
        attention_weights = torch.norm(utterance_features, dim=2)
        attention_weights = attention_weights.masked_fill(attention_masks, 0)
        attention_weights = F.softmax(attention_weights + (attention_masks.float() * -1e9), dim=1)
        
        return logits, attention_weights


# =============================================================================
# Dataset & Collate (음성과 유사, 특징만 변경)
# =============================================================================
class TextUtteranceDataset(Dataset):
    def __init__(self, participant_data):
        self.data = []
        for pid, info in participant_data.items():
            self.data.append({
                'pid': pid,
                'label': info['label'],
                'utterances': info['utterances'],
                'num_utterances': info['num_utterances']
            })
    
    def __len__(self):
        return len(self.data)
    
    def __getitem__(self, idx):
        return self.data[idx]


def collate_fn(batch):
    batch_labels = []
    all_utterances = []
    num_utterances_list = []
    batch_pids = []
    
    for item in batch:
        batch_labels.append(item['label'])
        all_utterances.extend(item['utterances'])
        num_utterances_list.append(item['num_utterances'])
        batch_pids.append(item['pid'])
    
    # 특징 추출
    batch_bert = []
    batch_ttrs = []
    batch_q_type_ids = []
    
    for utt in all_utterances:
        batch_bert.append(utt['bert'])
        batch_ttrs.append(utt['ttr'])
        batch_q_type_ids.append(utt['q_type_id'])
    
    # Tensor 변환
    batch_bert = torch.FloatTensor(np.array(batch_bert))  # [total_utterances, 768]
    batch_ttrs = torch.FloatTensor(batch_ttrs)  # [total_utterances]
    batch_q_type_ids = torch.LongTensor(batch_q_type_ids)  # [total_utterances]
    batch_labels = torch.FloatTensor(batch_labels).unsqueeze(1)  # [batch_size, 1]
    
    return {
        'batch_bert': batch_bert,
        'batch_ttrs': batch_ttrs,
        'batch_q_type_ids': batch_q_type_ids,
        'batch_labels': batch_labels,
        'num_utterances_list': num_utterances_list,
        'batch_pids': batch_pids
    }


# =============================================================================
# 데이터 로드 및 분할
# =============================================================================
def load_and_split_data():
    print(f"{'='*70}")
    print(f"📂 데이터 로드 중...")
    print(f"{'='*70}")
    
    with open(PREPROCESSED_DATA_PATH, 'rb') as f:
        dataset = pickle.load(f)
    
    # 메타데이터 로드하여 Group 정보 사용
    meta_df = pd.read_csv(os.path.join(BASE_PATH, "metadataset.csv"))
    meta_df['Participant_ID'] = meta_df['Participant_ID'].astype(str)
    
    # Group 열을 기준으로 분할
    train_pids = meta_df[meta_df['Group'] == 'Train']['Participant_ID'].tolist()
    val_pids = meta_df[meta_df['Group'] == 'Validation']['Participant_ID'].tolist()
    test_pids = meta_df[meta_df['Group'] == 'Test']['Participant_ID'].tolist()
    
    # dataset에 있는 PID만 사용 (전처리된 데이터 기준)
    train_pids = [pid for pid in train_pids if pid in dataset]
    val_pids = [pid for pid in val_pids if pid in dataset]
    test_pids = [pid for pid in test_pids if pid in dataset]
    
    # 라벨 추출
    train_labels = [dataset[pid]['label'] for pid in train_pids]
    val_labels = [dataset[pid]['label'] for pid in val_pids]
    test_labels = [dataset[pid]['label'] for pid in test_pids]
    
    print(f"총 참가자 수: {len(dataset)}")
    print(f"  - 정상 (0): {sum(1 for d in dataset.values() if d['label'] == 0)}")
    print(f"  - 우울증 (1): {sum(1 for d in dataset.values() if d['label'] == 1)}")
    
    print(f"\n메타데이터 기준 분할:")
    print(f"  - Train: {len(train_pids)} (0: {train_labels.count(0)}, 1: {train_labels.count(1)})")
    print(f"  - Val:   {len(val_pids)} (0: {val_labels.count(0)}, 1: {val_labels.count(1)})")
    print(f"  - Test:  {len(test_pids)} (0: {test_labels.count(0)}, 1: {test_labels.count(1)})")
    
    # 데이터셋 생성
    train_data = {pid: dataset[pid] for pid in train_pids}
    val_data = {pid: dataset[pid] for pid in val_pids}
    test_data = {pid: dataset[pid] for pid in test_pids}
    
    train_dataset = TextUtteranceDataset(train_data)
    val_dataset = TextUtteranceDataset(val_data)
    test_dataset = TextUtteranceDataset(test_data)
    
    # DataLoader 생성
    train_loader = DataLoader(
        train_dataset, batch_size=BATCH_SIZE, shuffle=True,
        collate_fn=collate_fn, num_workers=0, pin_memory=True if torch.cuda.is_available() else False
    )
    val_loader = DataLoader(
        val_dataset, batch_size=BATCH_SIZE, shuffle=False,
        collate_fn=collate_fn, num_workers=0, pin_memory=True if torch.cuda.is_available() else False
    )
    test_loader = DataLoader(
        test_dataset, batch_size=BATCH_SIZE, shuffle=False,
        collate_fn=collate_fn, num_workers=0, pin_memory=True if torch.cuda.is_available() else False
    )
    
    return train_loader, val_loader, test_loader


# =============================================================================
# 평가 함수 (음성 모델과 동일)
# =============================================================================
def evaluate(model, dataloader, criterion, threshold=0.5, return_all_thresholds=False):
    model.eval()
    
    all_labels = []
    all_probs = []
    all_preds = []
    total_loss = 0.0
    
    with torch.no_grad():
        for batch in dataloader:
            batch_bert = batch['batch_bert'].to(DEVICE)
            batch_ttrs = batch['batch_ttrs'].to(DEVICE)
            batch_q_type_ids = batch['batch_q_type_ids'].to(DEVICE)
            batch_labels = batch['batch_labels'].to(DEVICE)
            
            logits, _ = model(
                batch_bert,
                batch_ttrs,
                batch_q_type_ids,
                batch['num_utterances_list']
            )
            
            loss = criterion(logits, batch_labels)
            total_loss += loss.item()
            
            probs = torch.sigmoid(logits)
            preds = (probs > threshold).float()
            
            all_labels.extend(batch_labels.cpu().numpy().flatten())
            all_probs.extend(probs.cpu().numpy().flatten())
            all_preds.extend(preds.cpu().numpy().flatten())
    
    avg_loss = total_loss / len(dataloader)
    f1 = f1_score(all_labels, all_preds)
    precision = precision_score(all_labels, all_preds, zero_division=0)
    recall = recall_score(all_labels, all_preds, zero_division=0)
    
    return {
        'loss': avg_loss,
        'f1': f1,
        'precision': precision,
        'recall': recall,
        'labels': all_labels,
        'probs': all_probs,
        'preds': all_preds
    }


def find_optimal_threshold(labels, probs, min_recall=0.75):
    best_f1 = 0.0
    best_threshold = 0.5
    best_metrics = {}
    
    for thresh in np.arange(0.1, 0.7, 0.02):
        preds = (np.array(probs) > thresh).astype(int)
        f1 = f1_score(labels, preds)
        recall = recall_score(labels, preds, zero_division=0)
        precision = precision_score(labels, preds, zero_division=0)
        
        if recall < min_recall:
            continue
        
        if f1 > best_f1:
            best_f1 = f1
            best_threshold = thresh
            best_metrics = {'f1': f1, 'precision': precision, 'recall': recall, 'threshold': thresh}
    
    return best_threshold, best_metrics


# =============================================================================
# Training Loop (음성과 동일, forward만 수정)
# =============================================================================
def train_model(model, train_loader, val_loader, num_epochs=NUM_EPOCHS):
    # Loss Function 설정
    if USE_FOCAL_LOSS:
        criterion = FocalLoss(alpha=FOCAL_ALPHA, gamma=FOCAL_GAMMA, label_smoothing=LABEL_SMOOTHING)
        print(f"\n📍 Using Focal Loss (alpha={FOCAL_ALPHA}, gamma={FOCAL_GAMMA})")
    else:
        # Balanced BCE with pos_weight
        pos_weight = torch.tensor([POS_WEIGHT]).to(DEVICE)
        criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weight)
        print(f"\n📍 Using Balanced BCE Loss (pos_weight={POS_WEIGHT:.2f})")
    
    optimizer = optim.AdamW(model.parameters(), lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY)
    
    warmup_epochs = 5
    warmup_scheduler = optim.lr_scheduler.LinearLR(optimizer, start_factor=0.1, end_factor=1.0, total_iters=warmup_epochs)
    cosine_scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=num_epochs - warmup_epochs, eta_min=1e-6)
    scheduler = optim.lr_scheduler.SequentialLR(optimizer, schedulers=[warmup_scheduler, cosine_scheduler], milestones=[warmup_epochs])
    
    best_val_f1 = 0.0
    best_val_precision = 0.0
    best_val_recall = 0.0
    patience_counter = 0
    
    history = {'train_loss': [], 'val_loss': [], 'val_f1': [], 'val_precision': [], 'val_recall': []}
    
    print(f"\n{'='*70}")
    print(f"🚀 학습 시작")
    print(f"{'='*70}\n")
    
    for epoch in range(num_epochs):
        model.train()
        train_loss = 0.0
        
        pbar = tqdm(train_loader, desc=f"Epoch {epoch+1}/{num_epochs}")
        for batch in pbar:
            batch_bert = batch['batch_bert'].to(DEVICE)
            batch_ttrs = batch['batch_ttrs'].to(DEVICE)
            batch_q_type_ids = batch['batch_q_type_ids'].to(DEVICE)
            batch_labels = batch['batch_labels'].to(DEVICE)
            
            optimizer.zero_grad()
            logits, _ = model(batch_bert, batch_ttrs, batch_q_type_ids, batch['num_utterances_list'])
            
            loss = criterion(logits, batch_labels)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), GRADIENT_CLIP)
            optimizer.step()
            
            train_loss += loss.item()
            pbar.set_postfix({'loss': loss.item()})
        
        avg_train_loss = train_loss / len(train_loader)
        
        val_results = evaluate(model, val_loader, criterion, threshold=0.5)
        
        history['train_loss'].append(avg_train_loss)
        history['val_loss'].append(val_results['loss'])
        history['val_f1'].append(val_results['f1'])
        history['val_precision'].append(val_results['precision'])
        history['val_recall'].append(val_results['recall'])
        
        scheduler.step()
        
        print(f"\nEpoch {epoch+1}/{num_epochs}")
        print(f"  Train Loss: {avg_train_loss:.4f}")
        print(f"  Val Loss:   {val_results['loss']:.4f}")
        print(f"  Val F1:     {val_results['f1']:.4f}")
        print(f"  Val Prec:   {val_results['precision']:.4f}")
        print(f"  Val Recall: {val_results['recall']:.4f}")
        
        # 🔍 디버깅: 예측 분포 확인
        val_preds_sum = sum(val_results['preds'])
        val_labels_sum = sum(val_results['labels'])
        val_probs_array = np.array(val_results['probs'])
        print(f"  [DEBUG] Val 예측 분포: {val_preds_sum}/{len(val_results['preds'])} 우울증 예측")
        print(f"  [DEBUG] Val 실제 분포: {val_labels_sum}/{len(val_results['labels'])} 우울증 실제")
        print(f"  [DEBUG] Val 예측 확률: min={val_probs_array.min():.3f}, max={val_probs_array.max():.3f}, mean={val_probs_array.mean():.3f}")
        
        # 조건: F1 개선 또는 첫 epoch
        improved = val_results['f1'] > best_val_f1
        
        if improved or epoch == 0:  # 첫 epoch는 무조건 저장
            if improved:
                best_val_f1 = val_results['f1']
                best_val_precision = val_results['precision']
                best_val_recall = val_results['recall']
            
            patience_counter = 0
            
            torch.save({
                'epoch': epoch,
                'model_state_dict': model.state_dict(),
                'optimizer_state_dict': optimizer.state_dict(),
                'val_f1': val_results['f1'],
                'val_precision': val_results['precision'],
                'val_recall': val_results['recall'],
                'history': history
            }, os.path.join(BASE_PATH, 'best_text_model.pt'))
            
            if epoch == 0:
                print(f"  💾 초기 모델 저장 (F1: {val_results['f1']:.4f})")
            else:
                print(f"  ✅ Best model saved! (F1: {best_val_f1:.4f})")
        else:
            patience_counter += 1
            print(f"  ⏳ No improvement ({patience_counter}/{EARLY_STOPPING_PATIENCE})")
            
            if patience_counter >= EARLY_STOPPING_PATIENCE:
                print(f"\n⚠️  Early stopping!")
                break
        
        print("-" * 70)
    
    return history


# =============================================================================
# 메인 실행
# =============================================================================
if __name__ == "__main__":
    
    # 데이터 로드
    train_loader, val_loader, test_loader = load_and_split_data()
    
    # 모델 초기화
    model = TransformerTextDepressionModel(
        input_dim=INPUT_DIM,
        d_model=D_MODEL,
        nhead=NHEAD,
        num_encoder_layers=NUM_ENCODER_LAYERS,
        dim_feedforward=DIM_FEEDFORWARD,
        dropout=DROPOUT,
        q_type_vocab_size=Q_TYPE_VOCAB_SIZE,
        q_type_embed_dim=Q_TYPE_EMBED_DIM
    ).to(DEVICE)
    
    total_params = sum(p.numel() for p in model.parameters())
    print(f"\n{'='*70}")
    print(f"🤖 모델 초기화 완료")
    print(f"{'='*70}")
    print(f"총 파라미터 수: {total_params:,}")
    print(f"Device: {DEVICE}")
    print(f"{'='*70}\n")
    
    # 학습
    history = train_model(model, train_loader, val_loader)
    
    # Test 평가
    model_path = os.path.join(BASE_PATH, 'best_text_model.pt')
    
    if os.path.exists(model_path):
        print(f"\n{'='*70}")
        print(f"📊 Test Set 평가")
        print(f"{'='*70}\n")
        
        checkpoint = torch.load(model_path)
        model.load_state_dict(checkpoint['model_state_dict'])
        
        # 먼저 전체 확률 추출
        test_results_full = evaluate(model, test_loader, nn.BCEWithLogitsLoss(), threshold=0.5)
        all_probs = np.array(test_results_full['probs'])
        all_labels = np.array(test_results_full['labels'])
        
        # Threshold 최적화
        print("🎯 Threshold 최적화:")
        print("Threshold | F1    | Prec  | Rec   | Spec")
        print("-" * 50)
        
        best_f1 = 0.0
        best_threshold = 0.5
        best_metrics = {}
        
        for thresh in np.arange(0.2, 0.8, 0.05):
            preds = (all_probs > thresh).astype(int)
            
            f1 = f1_score(all_labels, preds)
            precision = precision_score(all_labels, preds, zero_division=0)
            recall = recall_score(all_labels, preds, zero_division=0)
            
            tn = sum((l == 0 and p == 0) for l, p in zip(all_labels, preds))
            fp = sum((l == 0 and p == 1) for l, p in zip(all_labels, preds))
            specificity = tn / (tn + fp) if (tn + fp) > 0 else 0
            
            if f1 > best_f1:
                best_f1 = f1
                best_threshold = thresh
                best_metrics = {
                    'f1': f1,
                    'precision': precision,
                    'recall': recall,
                    'specificity': specificity
                }
            
            marker = "⭐" if abs(thresh - best_threshold) < 0.01 else ""
            print(f"{thresh:.2f}      | {f1:.3f} | {precision:.3f} | {recall:.3f} | {specificity:.3f} {marker}")
        
        print(f"\n{'='*70}")
        print(f"💡 최적 Threshold: {best_threshold:.2f}")
        print(f"{'='*70}")
        print(f"최종 Test 성능 (Threshold={best_threshold:.2f}):")
        print(f"  F1:          {best_metrics['f1']:.4f}")
        print(f"  Precision:   {best_metrics['precision']:.4f}")
        print(f"  Recall:      {best_metrics['recall']:.4f}")
        print(f"  Specificity: {best_metrics['specificity']:.4f}")
        
        print(f"\n✅ 학습 및 평가 완료!")
        print(f"\n{'='*70}")
        print(f"📊 음성 vs 텍스트 비교")
        print(f"{'='*70}")
        print(f"음성 (Threshold=0.68):")
        print(f"  F1: 0.700, Precision: 0.583, Recall: 0.875")
        print(f"\n텍스트 (Threshold={best_threshold:.2f}):")
        print(f"  F1: {best_metrics['f1']:.3f}, Precision: {best_metrics['precision']:.3f}, Recall: {best_metrics['recall']:.3f}")
    else:
        print(f"\n⚠️ 모델 파일을 찾을 수 없습니다: {model_path}")
        print(f"   학습 중 early stopping이 발동하지 않았거나 에러가 발생했을 수 있습니다.")
        print(f"\n✅ 학습 완료!")

📝 텍스트 기반 우울증 감지 모델 학습
Input Dimension: 801
Device: cuda

📂 데이터 로드 중...
총 참가자 수: 186
  - 정상 (0): 129
  - 우울증 (1): 57

메타데이터 기준 분할:
  - Train: 107 (0: 76, 1: 31)
  - Val:   33 (0: 21, 1: 12)
  - Test:  46 (0: 32, 1: 14)

🤖 모델 초기화 완료
총 파라미터 수: 1,820,577
Device: cuda


📍 Using Balanced BCE Loss (pos_weight=2.45)

🚀 학습 시작



Epoch 1/100: 100%|██████████| 54/54 [00:01<00:00, 43.16it/s, loss=0.753]



Epoch 1/100
  Train Loss: 1.0246
  Val Loss:   1.0856
  Val F1:     0.0000
  Val Prec:   0.0000
  Val Recall: 0.0000
  [DEBUG] Val 예측 분포: 0.0/33 우울증 예측
  [DEBUG] Val 실제 분포: 12.0/33 우울증 실제
  [DEBUG] Val 예측 확률: min=0.440, max=0.468, mean=0.450
  💾 초기 모델 저장 (F1: 0.0000)
----------------------------------------------------------------------


Epoch 2/100: 100%|██████████| 54/54 [00:01<00:00, 48.66it/s, loss=0.609]



Epoch 2/100
  Train Loss: 1.0173
  Val Loss:   1.0978
  Val F1:     0.0000
  Val Prec:   0.0000
  Val Recall: 0.0000
  [DEBUG] Val 예측 분포: 0.0/33 우울증 예측
  [DEBUG] Val 실제 분포: 12.0/33 우울증 실제
  [DEBUG] Val 예측 확률: min=0.425, max=0.451, mean=0.433
  ⏳ No improvement (1/20)
----------------------------------------------------------------------


...


Epoch 19/100
  Train Loss: 1.4671
  Val Loss:   2.0427
  Val F1:     0.0000
  Val Prec:   0.0000
  Val Recall: 0.0000
  [DEBUG] Val 예측 분포: 0.0/33 우울증 예측
  [DEBUG] Val 실제 분포: 12.0/33 우울증 실제
  [DEBUG] Val 예측 확률: min=0.098, max=0.106, mean=0.101
  ⏳ No improvement (18/20)
----------------------------------------------------------------------


Epoch 20/100: 100%|██████████| 54/54 [00:01<00:00, 51.45it/s, loss=0.108] 



Epoch 20/100
  Train Loss: 1.5529
  Val Loss:   2.1136
  Val F1:     0.0000
  Val Prec:   0.0000
  Val Recall: 0.0000
  [DEBUG] Val 예측 분포: 0.0/33 우울증 예측
  [DEBUG] Val 실제 분포: 12.0/33 우울증 실제
  [DEBUG] Val 예측 확률: min=0.089, max=0.097, mean=0.092
  ⏳ No improvement (19/20)
----------------------------------------------------------------------


Epoch 21/100: 100%|██████████| 54/54 [00:01<00:00, 50.58it/s, loss=4.29] 



Epoch 21/100
  Train Loss: 1.5744
  Val Loss:   2.2051
  Val F1:     0.0000
  Val Prec:   0.0000
  Val Recall: 0.0000
  [DEBUG] Val 예측 분포: 0.0/33 우울증 예측
  [DEBUG] Val 실제 분포: 12.0/33 우울증 실제
  [DEBUG] Val 예측 확률: min=0.079, max=0.087, mean=0.082
  ⏳ No improvement (20/20)

⚠️  Early stopping!

📊 Test Set 평가

🎯 Threshold 최적화:
Threshold | F1    | Prec  | Rec   | Spec
--------------------------------------------------
0.20      | 0.467 | 0.304 | 1.000 | 0.000 ⭐
0.25      | 0.467 | 0.304 | 1.000 | 0.000 
0.30      | 0.467 | 0.304 | 1.000 | 0.000 
0.35      | 0.467 | 0.304 | 1.000 | 0.000 
0.40      | 0.467 | 0.304 | 1.000 | 0.000 
0.45      | 0.308 | 0.240 | 0.429 | 0.406 
0.50      | 0.000 | 0.000 | 0.000 | 1.000 
0.55      | 0.000 | 0.000 | 0.000 | 1.000 
0.60      | 0.000 | 0.000 | 0.000 | 1.000 
0.65      | 0.000 | 0.000 | 0.000 | 1.000 
0.70      | 0.000 | 0.000 | 0.000 | 1.000 
0.75      | 0.000 | 0.000 | 0.000 | 1.000 
0.80      | 0.000 | 0.000 | 0.000 | 1.000 

💡 최적 Threshold: 0.20
최

In [27]:
"""
텍스트 모델 Threshold 최적화 (저장된 모델 사용)
"""
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import pickle
import os
import math
from sklearn.model_selection import train_test_split
from sklearn.metrics import f1_score, precision_score, recall_score
from torch.utils.data import Dataset, DataLoader

# =============================================================================
# 설정
# =============================================================================
BASE_PATH = r"D:\depression_dataset(DAIC-WOZ)"
MODEL_PATH = os.path.join(BASE_PATH, "best_text_model.pt")
DATA_PATH = os.path.join(BASE_PATH, "preprocessed_text_dataset.pkl")

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

print(f"{'='*70}")
print(f"🎯 텍스트 모델 Threshold 최적화")
print(f"{'='*70}\n")


# =============================================================================
# 모델 정의 (text_model_training.py와 동일)
# =============================================================================
class PositionalEncoding(nn.Module):
    def __init__(self, d_model, max_len=500, dropout=0.1):
        super(PositionalEncoding, self).__init__()
        self.dropout = nn.Dropout(p=dropout)
        
        pe = torch.zeros(max_len, d_model)
        position = torch.arange(0, max_len, dtype=torch.float).unsqueeze(1)
        div_term = torch.exp(torch.arange(0, d_model, 2).float() * (-math.log(10000.0) / d_model))
        
        pe[:, 0::2] = torch.sin(position * div_term)
        pe[:, 1::2] = torch.cos(position * div_term)
        pe = pe.unsqueeze(0)
        
        self.register_buffer('pe', pe)
    
    def forward(self, x):
        x = x + self.pe[:, :x.size(1), :]
        return self.dropout(x)


class TransformerTextDepressionModel(nn.Module):
    def __init__(self, input_dim=801, d_model=256, nhead=8, num_encoder_layers=3,
                 dim_feedforward=512, dropout=0.3, q_type_vocab_size=5, q_type_embed_dim=32):
        super(TransformerTextDepressionModel, self).__init__()
        
        self.d_model = d_model
        self.q_type_embedding = nn.Embedding(q_type_vocab_size, q_type_embed_dim)
        self.input_projection = nn.Linear(input_dim, d_model)
        self.pos_encoder = PositionalEncoding(d_model, dropout=dropout)
        
        encoder_layer = nn.TransformerEncoderLayer(
            d_model=d_model, nhead=nhead, dim_feedforward=dim_feedforward,
            dropout=dropout, activation='gelu', batch_first=True
        )
        
        self.transformer_encoder = nn.TransformerEncoder(encoder_layer, num_layers=num_encoder_layers)
        self.cls_token = nn.Parameter(torch.randn(1, 1, d_model))
        
        self.classifier = nn.Sequential(
            nn.LayerNorm(d_model), nn.Dropout(dropout),
            nn.Linear(d_model, d_model // 2), nn.GELU(),
            nn.Dropout(dropout), nn.Linear(d_model // 2, 1)
        )
        
        self.dropout = nn.Dropout(dropout)
    
    def forward(self, batch_bert, batch_ttrs, batch_q_type_ids, num_utterances_list):
        batch_size = len(num_utterances_list)
        device = batch_bert.device
        
        q_type_embs = self.q_type_embedding(batch_q_type_ids)
        ttrs_expanded = batch_ttrs.unsqueeze(1)
        
        combined_features = torch.cat([batch_bert, q_type_embs, ttrs_expanded], dim=1)
        
        participant_sequences = []
        start_idx = 0
        
        for num_utts in num_utterances_list:
            end_idx = start_idx + num_utts
            participant_sequences.append(combined_features[start_idx:end_idx])
            start_idx = end_idx
        
        max_num_utterances = max(num_utterances_list)
        padded_sequences = []
        attention_masks = []
        
        for seq, num_utts in zip(participant_sequences, num_utterances_list):
            if num_utts < max_num_utterances:
                padding = torch.zeros(max_num_utterances - num_utts, seq.size(1), device=device)
                seq = torch.cat([seq, padding], dim=0)
            
            padded_sequences.append(seq)
            
            mask = torch.zeros(max_num_utterances, dtype=torch.bool, device=device)
            mask[num_utts:] = True
            attention_masks.append(mask)
        
        padded_sequences = torch.stack(padded_sequences)
        attention_masks = torch.stack(attention_masks)
        
        x = self.input_projection(padded_sequences)
        cls_tokens = self.cls_token.expand(batch_size, -1, -1)
        x = torch.cat([cls_tokens, x], dim=1)
        
        cls_mask = torch.zeros(batch_size, 1, dtype=torch.bool, device=device)
        extended_mask = torch.cat([cls_mask, attention_masks], dim=1)
        
        x = self.pos_encoder(x)
        encoded = self.transformer_encoder(x, src_key_padding_mask=extended_mask)
        
        cls_output = encoded[:, 0, :]
        logits = self.classifier(cls_output)
        
        utterance_features = encoded[:, 1:, :]
        attention_weights = torch.norm(utterance_features, dim=2)
        attention_weights = attention_weights.masked_fill(attention_masks, 0)
        attention_weights = F.softmax(attention_weights + (attention_masks.float() * -1e9), dim=1)
        
        return logits, attention_weights


# =============================================================================
# Dataset
# =============================================================================
class TextUtteranceDataset(Dataset):
    def __init__(self, participant_data):
        self.data = []
        for pid, info in participant_data.items():
            self.data.append({
                'pid': pid, 'label': info['label'],
                'utterances': info['utterances'],
                'num_utterances': info['num_utterances']
            })
    
    def __len__(self):
        return len(self.data)
    
    def __getitem__(self, idx):
        return self.data[idx]


def collate_fn(batch):
    batch_labels = []
    all_utterances = []
    num_utterances_list = []
    batch_pids = []
    
    for item in batch:
        batch_labels.append(item['label'])
        all_utterances.extend(item['utterances'])
        num_utterances_list.append(item['num_utterances'])
        batch_pids.append(item['pid'])
    
    batch_bert = []
    batch_ttrs = []
    batch_q_type_ids = []
    
    for utt in all_utterances:
        batch_bert.append(utt['bert'])
        batch_ttrs.append(utt['ttr'])
        batch_q_type_ids.append(utt['q_type_id'])
    
    batch_bert = torch.FloatTensor(np.array(batch_bert))
    batch_ttrs = torch.FloatTensor(batch_ttrs)
    batch_q_type_ids = torch.LongTensor(batch_q_type_ids)
    batch_labels = torch.FloatTensor(batch_labels).unsqueeze(1)
    
    return {
        'batch_bert': batch_bert, 'batch_ttrs': batch_ttrs,
        'batch_q_type_ids': batch_q_type_ids, 'batch_labels': batch_labels,
        'num_utterances_list': num_utterances_list, 'batch_pids': batch_pids
    }


# =============================================================================
# 데이터 로드
# =============================================================================
print("⏳ 데이터 로드 중...")
with open(DATA_PATH, 'rb') as f:
    dataset = pickle.load(f)

pids = list(dataset.keys())
labels = [dataset[pid]['label'] for pid in pids]

train_pids, temp_pids, train_labels, temp_labels = train_test_split(
    pids, labels, test_size=0.3, stratify=labels, random_state=42
)

val_pids, test_pids, val_labels, test_labels = train_test_split(
    temp_pids, temp_labels, test_size=0.5, stratify=temp_labels, random_state=42
)

test_data = {pid: dataset[pid] for pid in test_pids}
test_dataset = TextUtteranceDataset(test_data)
test_loader = DataLoader(test_dataset, batch_size=4, shuffle=False, collate_fn=collate_fn)

print(f"✅ Test set: {len(test_pids)}명 (정상: {test_labels.count(0)}, 우울증: {test_labels.count(1)})")


# =============================================================================
# 모델 로드
# =============================================================================
print("\n⏳ 모델 로드 중...")
checkpoint = torch.load(MODEL_PATH)

model = TransformerTextDepressionModel(
    input_dim=801, d_model=256, nhead=8, num_encoder_layers=3,
    dim_feedforward=512, dropout=0.3, q_type_vocab_size=5, q_type_embed_dim=32
).to(DEVICE)

model.load_state_dict(checkpoint['model_state_dict'])
model.eval()

print(f"✅ Best epoch: {checkpoint['epoch']}, Val F1: {checkpoint['val_f1']:.4f}")


# =============================================================================
# 예측 확률 추출
# =============================================================================
print("\n⏳ 예측 수행 중...")

all_labels = []
all_probs = []

with torch.no_grad():
    for batch in test_loader:
        batch_bert = batch['batch_bert'].to(DEVICE)
        batch_ttrs = batch['batch_ttrs'].to(DEVICE)
        batch_q_type_ids = batch['batch_q_type_ids'].to(DEVICE)
        batch_labels = batch['batch_labels'].to(DEVICE)
        
        logits, _ = model(batch_bert, batch_ttrs, batch_q_type_ids, batch['num_utterances_list'])
        probs = torch.sigmoid(logits).cpu().numpy().flatten()
        labels = batch_labels.cpu().numpy().flatten()
        
        all_labels.extend(labels)
        all_probs.extend(probs)

all_labels = np.array(all_labels)
all_probs = np.array(all_probs)

print(f"✅ 예측 완료: {len(all_labels)}명")
print(f"   예측 확률 범위: {all_probs.min():.3f} ~ {all_probs.max():.3f}")


# =============================================================================
# Threshold 최적화
# =============================================================================
print(f"\n{'='*70}")
print(f"🎯 Threshold 최적화")
print(f"{'='*70}\n")

print("Threshold | F1    | Prec  | Rec   | Spec  | TP | FP | FN | TN")
print("-" * 70)

best_f1 = 0.0
best_threshold = 0.5
best_metrics = {}

for thresh in np.arange(0.2, 0.8, 0.05):
    preds = (all_probs > thresh).astype(int)
    
    f1 = f1_score(all_labels, preds)
    precision = precision_score(all_labels, preds, zero_division=0)
    recall = recall_score(all_labels, preds, zero_division=0)
    
    tn = sum((l == 0 and p == 0) for l, p in zip(all_labels, preds))
    fp = sum((l == 0 and p == 1) for l, p in zip(all_labels, preds))
    fn = sum((l == 1 and p == 0) for l, p in zip(all_labels, preds))
    tp = sum((l == 1 and p == 1) for l, p in zip(all_labels, preds))
    
    specificity = tn / (tn + fp) if (tn + fp) > 0 else 0
    
    if f1 > best_f1:
        best_f1 = f1
        best_threshold = thresh
        best_metrics = {
            'f1': f1,
            'precision': precision,
            'recall': recall,
            'specificity': specificity,
            'tp': tp, 'fp': fp, 'fn': fn, 'tn': tn
        }
    
    marker = "⭐" if abs(thresh - best_threshold) < 0.01 else ""
    print(f"{thresh:.2f}      | {f1:.3f} | {precision:.3f} | {recall:.3f} | {specificity:.3f} | {int(tp):2d} | {int(fp):2d} | {int(fn):2d} | {int(tn):2d} {marker}")


# =============================================================================
# 최종 결과
# =============================================================================
print(f"\n{'='*70}")
print(f"💡 최적 Threshold: {best_threshold:.2f}")
print(f"{'='*70}")
print(f"\n최종 Test 성능 (Threshold={best_threshold:.2f}):")
print(f"  F1:          {best_metrics['f1']:.4f}")
print(f"  Precision:   {best_metrics['precision']:.4f}")
print(f"  Recall:      {best_metrics['recall']:.4f}")
print(f"  Specificity: {best_metrics['specificity']:.4f}")
print(f"\n  TP: {int(best_metrics['tp'])}, FP: {int(best_metrics['fp'])}, FN: {int(best_metrics['fn'])}, TN: {int(best_metrics['tn'])}")

print(f"\n{'='*70}")
print(f"📊 음성 vs 텍스트 비교")
print(f"{'='*70}")
print(f"\n음성 모델 (Threshold=0.68):")
print(f"  F1: 0.700, Precision: 0.583, Recall: 0.875, Specificity: 0.750")
print(f"\n텍스트 모델 (Threshold={best_threshold:.2f}):")
print(f"  F1: {best_metrics['f1']:.3f}, Precision: {best_metrics['precision']:.3f}, Recall: {best_metrics['recall']:.3f}, Specificity: {best_metrics['specificity']:.3f}")

print(f"\n{'='*70}")
print(f"✅ 분석 완료!")
print(f"{'='*70}\n")

🎯 텍스트 모델 Threshold 최적화

⏳ 데이터 로드 중...
✅ Test set: 28명 (정상: 20, 우울증: 8)

⏳ 모델 로드 중...
✅ Best epoch: 0, Val F1: 0.0000

⏳ 예측 수행 중...
✅ 예측 완료: 28명
   예측 확률 범위: 0.438 ~ 0.463

🎯 Threshold 최적화

Threshold | F1    | Prec  | Rec   | Spec  | TP | FP | FN | TN
----------------------------------------------------------------------
0.20      | 0.444 | 0.286 | 1.000 | 0.000 |  8 | 20 |  0 |  0 ⭐
0.25      | 0.444 | 0.286 | 1.000 | 0.000 |  8 | 20 |  0 |  0 
0.30      | 0.444 | 0.286 | 1.000 | 0.000 |  8 | 20 |  0 |  0 
0.35      | 0.444 | 0.286 | 1.000 | 0.000 |  8 | 20 |  0 |  0 
0.40      | 0.444 | 0.286 | 1.000 | 0.000 |  8 | 20 |  0 |  0 
0.45      | 0.250 | 0.250 | 0.250 | 0.700 |  2 |  6 |  6 | 14 
0.50      | 0.000 | 0.000 | 0.000 | 1.000 |  0 |  0 |  8 | 20 
0.55      | 0.000 | 0.000 | 0.000 | 1.000 |  0 |  0 |  8 | 20 
0.60      | 0.000 | 0.000 | 0.000 | 1.000 |  0 |  0 |  8 | 20 
0.65      | 0.000 | 0.000 | 0.000 | 1.000 |  0 |  0 |  8 | 20 
0.70      | 0.000 | 0.000 | 0.000 | 1.000 |  0 

CHECK MODALITIES POSIBILITIES

In [28]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import pickle
import os
import math
from collections import defaultdict
from sklearn.metrics import f1_score, precision_score, recall_score, confusion_matrix

# =============================================================================
# Settings
# =============================================================================
BASE_PATH = r"D:\depression_dataset(DAIC-WOZ)"
AUDIO_MODEL_PATH = os.path.join(BASE_PATH, "best_transformer_model.pt")
TEXT_MODEL_PATH = os.path.join(BASE_PATH, "best_text_model.pt")
AUDIO_DATA_PATH = os.path.join(BASE_PATH, "preprocessed_utterance_dataset.pkl")
TEXT_DATA_PATH = os.path.join(BASE_PATH, "preprocessed_text_dataset.pkl")

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

# Optimal thresholds from individual model training
AUDIO_THRESHOLD = 0.68
TEXT_THRESHOLD = 0.20


# =============================================================================
# Model Definition (analyze_dual_attention.py와 동일)
# =============================================================================
class PositionalEncoding(nn.Module):
    def __init__(self, d_model, max_len=500, dropout=0.1):
        super(PositionalEncoding, self).__init__()
        self.dropout = nn.Dropout(p=dropout)
        pe = torch.zeros(max_len, d_model)
        position = torch.arange(0, max_len, dtype=torch.float).unsqueeze(1)
        div_term = torch.exp(torch.arange(0, d_model, 2).float() * (-math.log(10000.0) / d_model))
        pe[:, 0::2] = torch.sin(position * div_term)
        pe[:, 1::2] = torch.cos(position * div_term)
        pe = pe.unsqueeze(0)
        self.register_buffer('pe', pe)
    
    def forward(self, x):
        x = x + self.pe[:, :x.size(1), :]
        return self.dropout(x)


class TransformerDepressionModel(nn.Module):
    """Audio Model"""
    def __init__(self, input_dim=801, d_model=256, nhead=8, num_encoder_layers=3,
                 dim_feedforward=512, dropout=0.3, q_type_vocab_size=5, q_type_embed_dim=32):
        super(TransformerDepressionModel, self).__init__()
        
        self.d_model = d_model
        self.q_type_embedding = nn.Embedding(q_type_vocab_size, q_type_embed_dim)
        self.input_projection = nn.Linear(input_dim, d_model)
        self.pos_encoder = PositionalEncoding(d_model, dropout=dropout)
        
        encoder_layer = nn.TransformerEncoderLayer(
            d_model=d_model, nhead=nhead, dim_feedforward=dim_feedforward,
            dropout=dropout, activation='gelu', batch_first=True
        )
        
        self.transformer_encoder = nn.TransformerEncoder(encoder_layer, num_layers=num_encoder_layers)
        self.cls_token = nn.Parameter(torch.randn(1, 1, d_model))
        
        self.classifier = nn.Sequential(
            nn.LayerNorm(d_model), nn.Dropout(dropout),
            nn.Linear(d_model, d_model // 2), nn.GELU(),
            nn.Dropout(dropout), nn.Linear(d_model // 2, 1)
        )
        
        self.dropout = nn.Dropout(dropout)
    
    def forward(self, batch_wav2vec, batch_ttrs, batch_q_type_ids, num_utterances_list):
        batch_size = len(num_utterances_list)
        device = batch_wav2vec.device
        
        q_type_embs = self.q_type_embedding(batch_q_type_ids)
        ttrs_expanded = batch_ttrs.unsqueeze(1)
        
        combined_features = torch.cat([batch_wav2vec, q_type_embs, ttrs_expanded], dim=1)
        
        participant_sequences = []
        start_idx = 0
        
        for num_utts in num_utterances_list:
            end_idx = start_idx + num_utts
            participant_sequences.append(combined_features[start_idx:end_idx])
            start_idx = end_idx
        
        max_num_utterances = max(num_utterances_list)
        padded_sequences = []
        attention_masks = []
        
        for seq, num_utts in zip(participant_sequences, num_utterances_list):
            if num_utts < max_num_utterances:
                padding = torch.zeros(max_num_utterances - num_utts, seq.size(1), device=device)
                seq = torch.cat([seq, padding], dim=0)
            
            padded_sequences.append(seq)
            
            mask = torch.zeros(max_num_utterances, dtype=torch.bool, device=device)
            mask[num_utts:] = True
            attention_masks.append(mask)
        
        padded_sequences = torch.stack(padded_sequences)
        attention_masks = torch.stack(attention_masks)
        
        x = self.input_projection(padded_sequences)
        cls_tokens = self.cls_token.expand(batch_size, -1, -1)
        x = torch.cat([cls_tokens, x], dim=1)
        
        cls_mask = torch.zeros(batch_size, 1, dtype=torch.bool, device=device)
        extended_mask = torch.cat([cls_mask, attention_masks], dim=1)
        
        x = self.pos_encoder(x)
        encoded = self.transformer_encoder(x, src_key_padding_mask=extended_mask)
        
        cls_output = encoded[:, 0, :]
        logits = self.classifier(cls_output)
        
        utterance_features = encoded[:, 1:, :]
        attention_weights = torch.norm(utterance_features, dim=2)
        attention_weights = attention_weights.masked_fill(attention_masks, 0)
        attention_weights = F.softmax(attention_weights + (attention_masks.float() * -1e9), dim=1)
        
        return logits, attention_weights


class TransformerTextDepressionModel(nn.Module):
    """Text Model"""
    def __init__(self, input_dim=801, d_model=256, nhead=8, num_encoder_layers=3,
                 dim_feedforward=512, dropout=0.3, q_type_vocab_size=5, q_type_embed_dim=32):
        super(TransformerTextDepressionModel, self).__init__()
        
        self.d_model = d_model
        self.q_type_embedding = nn.Embedding(q_type_vocab_size, q_type_embed_dim)
        self.input_projection = nn.Linear(input_dim, d_model)
        self.pos_encoder = PositionalEncoding(d_model, dropout=dropout)
        
        encoder_layer = nn.TransformerEncoderLayer(
            d_model=d_model, nhead=nhead, dim_feedforward=dim_feedforward,
            dropout=dropout, activation='gelu', batch_first=True
        )
        
        self.transformer_encoder = nn.TransformerEncoder(encoder_layer, num_layers=num_encoder_layers)
        self.cls_token = nn.Parameter(torch.randn(1, 1, d_model))
        
        self.classifier = nn.Sequential(
            nn.LayerNorm(d_model), nn.Dropout(dropout),
            nn.Linear(d_model, d_model // 2), nn.GELU(),
            nn.Dropout(dropout), nn.Linear(d_model // 2, 1)
        )
        
        self.dropout = nn.Dropout(dropout)
    
    def forward(self, batch_bert, batch_ttrs, batch_q_type_ids, num_utterances_list):
        batch_size = len(num_utterances_list)
        device = batch_bert.device
        
        q_type_embs = self.q_type_embedding(batch_q_type_ids)
        ttrs_expanded = batch_ttrs.unsqueeze(1)
        
        combined_features = torch.cat([batch_bert, q_type_embs, ttrs_expanded], dim=1)
        
        participant_sequences = []
        start_idx = 0
        
        for num_utts in num_utterances_list:
            end_idx = start_idx + num_utts
            participant_sequences.append(combined_features[start_idx:end_idx])
            start_idx = end_idx
        
        max_num_utterances = max(num_utterances_list)
        padded_sequences = []
        attention_masks = []
        
        for seq, num_utts in zip(participant_sequences, num_utterances_list):
            if num_utts < max_num_utterances:
                padding = torch.zeros(max_num_utterances - num_utts, seq.size(1), device=device)
                seq = torch.cat([seq, padding], dim=0)
            
            padded_sequences.append(seq)
            
            mask = torch.zeros(max_num_utterances, dtype=torch.bool, device=device)
            mask[num_utts:] = True
            attention_masks.append(mask)
        
        padded_sequences = torch.stack(padded_sequences)
        attention_masks = torch.stack(attention_masks)
        
        x = self.input_projection(padded_sequences)
        cls_tokens = self.cls_token.expand(batch_size, -1, -1)
        x = torch.cat([cls_tokens, x], dim=1)
        
        cls_mask = torch.zeros(batch_size, 1, dtype=torch.bool, device=device)
        extended_mask = torch.cat([cls_mask, attention_masks], dim=1)
        
        x = self.pos_encoder(x)
        encoded = self.transformer_encoder(x, src_key_padding_mask=extended_mask)
        
        cls_output = encoded[:, 0, :]
        logits = self.classifier(cls_output)
        
        utterance_features = encoded[:, 1:, :]
        attention_weights = torch.norm(utterance_features, dim=2)
        attention_weights = attention_weights.masked_fill(attention_masks, 0)
        attention_weights = F.softmax(attention_weights + (attention_masks.float() * -1e9), dim=1)
        
        return logits, attention_weights


# =============================================================================
# Data Extraction
# =============================================================================
def extract_predictions_from_models(audio_model, text_model, audio_data, text_data):
    """Extract predictions from both models for all common participants"""
    audio_model.eval()
    text_model.eval()
    
    results = []
    common_pids = set(audio_data.keys()) & set(text_data.keys())
    
    with torch.no_grad():
        for pid in sorted(common_pids):
            audio_info = audio_data[pid]
            text_info = text_data[pid]
            
            label = audio_info['label']
            
            # Audio forward
            audio_utterances = audio_info['utterances']
            wav2vec_features = torch.FloatTensor([utt['wav2vec'] for utt in audio_utterances]).to(DEVICE)
            audio_ttrs = torch.FloatTensor([utt['ttr'] for utt in audio_utterances]).to(DEVICE)
            audio_q_type_ids = torch.LongTensor([utt['q_type_id'] for utt in audio_utterances]).to(DEVICE)
            
            audio_logits, _ = audio_model(
                wav2vec_features, audio_ttrs, audio_q_type_ids, [len(audio_utterances)]
            )
            
            # Text forward
            text_utterances = text_info['utterances']
            bert_features = torch.FloatTensor([utt['bert'] for utt in text_utterances]).to(DEVICE)
            text_ttrs = torch.FloatTensor([utt['ttr'] for utt in text_utterances]).to(DEVICE)
            text_q_type_ids = torch.LongTensor([utt['q_type_id'] for utt in text_utterances]).to(DEVICE)
            
            text_logits, _ = text_model(
                bert_features, text_ttrs, text_q_type_ids, [len(text_utterances)]
            )
            
            audio_prob = torch.sigmoid(audio_logits).item()
            text_prob = torch.sigmoid(text_logits).item()
            
            results.append({
                'pid': pid,
                'label': label,
                'audio_prob': audio_prob,
                'text_prob': text_prob
            })
    
    return results


def split_data_by_metadata_group(results, base_path):
    """Split data into train/val/test based on metadataset.csv groups"""
    # 메타데이터 로드
    meta_df = pd.read_csv(os.path.join(base_path, "metadataset.csv"))
    meta_df['Participant_ID'] = meta_df['Participant_ID'].astype(str)
    
    # PID를 문자열로 변환
    results_df = pd.DataFrame(results)
    results_df['pid'] = results_df['pid'].astype(str)
    
    # 메타데이터와 병합
    results_df = results_df.merge(
        meta_df[['Participant_ID', 'Group']], 
        left_on='pid', 
        right_on='Participant_ID', 
        how='left'
    )
    
    # Group별로 분할
    train_data = results_df[results_df['Group'] == 'Train'].to_dict('records')
    val_data = results_df[results_df['Group'] == 'Validation'].to_dict('records')
    test_data = results_df[results_df['Group'] == 'Test'].to_dict('records')
    
    # 라벨 분포 계산
    def count_labels(data):
        labels = [d['label'] for d in data]
        return labels.count(0), labels.count(1)
    
    train_normal, train_dep = count_labels(train_data)
    val_normal, val_dep = count_labels(val_data)
    test_normal, test_dep = count_labels(test_data)
    
    print(f"\nMetadata-based split:")
    print(f"  Train: {len(train_data)} (Normal: {train_normal}, Depression: {train_dep})")
    print(f"  Val:   {len(val_data)} (Normal: {val_normal}, Depression: {val_dep})")
    print(f"  Test:  {len(test_data)} (Normal: {test_normal}, Depression: {test_dep})")
    
    return train_data, val_data, test_data


# =============================================================================
# Late Fusion with Threshold Correction
# =============================================================================
def evaluate_late_fusion(data, audio_weight, audio_threshold=0.3, text_threshold=0.2):
    """
    Threshold-aware late fusion evaluation
    
    Method 1: Probability fusion (기존 방식)
    Method 2: Decision fusion (threshold 적용 후 voting)
    """
    text_weight = 1 - audio_weight
    
    y_true = []
    y_pred_prob = []
    y_pred_decision = []
    
    for sample in data:
        label = sample['label']
        audio_prob = sample['audio_prob']
        text_prob = sample['text_prob']
        
        y_true.append(label)
        
        # Method 1: Weighted probability fusion
        fused_prob = audio_weight * audio_prob + text_weight * text_prob
        pred_prob = 1 if fused_prob > 0.5 else 0
        y_pred_prob.append(pred_prob)
        
        # Method 2: Decision-level fusion (threshold-aware)
        audio_decision = 1 if audio_prob > audio_threshold else 0
        text_decision = 1 if text_prob > text_threshold else 0
        
        # Weighted voting
        fused_decision = audio_weight * audio_decision + text_weight * text_decision
        pred_decision = 1 if fused_decision > 0.5 else 0
        y_pred_decision.append(pred_decision)
    
    # Calculate metrics
    f1_prob = f1_score(y_true, y_pred_prob, zero_division=0)
    precision_prob = precision_score(y_true, y_pred_prob, zero_division=0)
    recall_prob = recall_score(y_true, y_pred_prob, zero_division=0)
    
    f1_decision = f1_score(y_true, y_pred_decision, zero_division=0)
    precision_decision = precision_score(y_true, y_pred_decision, zero_division=0)
    recall_decision = recall_score(y_true, y_pred_decision, zero_division=0)
    
    # Calculate specificity
    cm_prob = confusion_matrix(y_true, y_pred_prob)
    cm_decision = confusion_matrix(y_true, y_pred_decision)
    
    if cm_prob.shape[0] == 2:
        tn_prob, fp_prob, fn_prob, tp_prob = cm_prob.ravel()
        spec_prob = tn_prob / (tn_prob + fp_prob) if (tn_prob + fp_prob) > 0 else 0
    else:
        spec_prob = 0
    
    if cm_decision.shape[0] == 2:
        tn_decision, fp_decision, fn_decision, tp_decision = cm_decision.ravel()
        spec_decision = tn_decision / (tn_decision + fp_decision) if (tn_decision + fp_decision) > 0 else 0
    else:
        spec_decision = 0
    
    return {
        'prob': {
            'f1': f1_prob, 
            'precision': precision_prob, 
            'recall': recall_prob,
            'specificity': spec_prob
        },
        'decision': {
            'f1': f1_decision, 
            'precision': precision_decision, 
            'recall': recall_decision,
            'specificity': spec_decision
        }
    }


def find_optimal_fusion_weight(train_data, val_data, audio_threshold=0.3, text_threshold=0.2):
    """Find optimal fusion weight using validation set"""
    
    print(f"\n{'='*70}")
    print(f"Late Fusion Weight Optimization (Threshold-Aware)")
    print(f"{'='*70}")
    print(f"Audio Threshold: {audio_threshold}")
    print(f"Text Threshold:  {text_threshold}\n")
    
    weights = np.arange(0, 1.05, 0.05)
    
    results_prob = []
    results_decision = []
    
    for w_audio in weights:
        metrics = evaluate_late_fusion(val_data, w_audio, audio_threshold, text_threshold)
        results_prob.append(metrics['prob'])
        results_decision.append(metrics['decision'])
    
    # Find best weights
    f1_scores_prob = [r['f1'] for r in results_prob]
    f1_scores_decision = [r['f1'] for r in results_decision]
    
    best_idx_prob = np.argmax(f1_scores_prob)
    best_idx_decision = np.argmax(f1_scores_decision)
    
    best_weight_prob = weights[best_idx_prob]
    best_weight_decision = weights[best_idx_decision]
    
    print(f"[Method 1: Probability Fusion]")
    print(f"  Best Audio Weight: {best_weight_prob:.2f}")
    print(f"  Best F1:           {f1_scores_prob[best_idx_prob]:.4f}")
    print(f"  Precision:         {results_prob[best_idx_prob]['precision']:.4f}")
    print(f"  Recall:            {results_prob[best_idx_prob]['recall']:.4f}")
    print(f"  Specificity:       {results_prob[best_idx_prob]['specificity']:.4f}")
    
    print(f"\n[Method 2: Decision Fusion (Threshold-Aware)]")
    print(f"  Best Audio Weight: {best_weight_decision:.2f}")
    print(f"  Best F1:           {f1_scores_decision[best_idx_decision]:.4f}")
    print(f"  Precision:         {results_decision[best_idx_decision]['precision']:.4f}")
    print(f"  Recall:            {results_decision[best_idx_decision]['recall']:.4f}")
    print(f"  Specificity:       {results_decision[best_idx_decision]['specificity']:.4f}")
    
    # Visualization
    fig, axes = plt.subplots(1, 2, figsize=(16, 6))
    
    # Method 1
    ax1 = axes[0]
    ax1.plot(weights, f1_scores_prob, 'o-', linewidth=2, markersize=6, color='blue', label='F1')
    ax1.plot(weights, [r['precision'] for r in results_prob], 's--', linewidth=1.5, markersize=4, 
             color='green', alpha=0.7, label='Precision')
    ax1.plot(weights, [r['recall'] for r in results_prob], '^--', linewidth=1.5, markersize=4, 
             color='red', alpha=0.7, label='Recall')
    ax1.plot(weights, [r['specificity'] for r in results_prob], 'd--', linewidth=1.5, markersize=4, 
             color='orange', alpha=0.7, label='Specificity')
    ax1.axvline(best_weight_prob, color='purple', linestyle='--', linewidth=2, 
                label=f'Best: {best_weight_prob:.2f}')
    ax1.set_xlabel('Audio Weight (Text = 1 - Audio)', fontsize=12)
    ax1.set_ylabel('Score', fontsize=12)
    ax1.set_title('Method 1: Probability Fusion', fontsize=14, fontweight='bold')
    ax1.legend()
    ax1.grid(alpha=0.3)
    
    # Method 2
    ax2 = axes[1]
    ax2.plot(weights, f1_scores_decision, 'o-', linewidth=2, markersize=6, color='blue', label='F1')
    ax2.plot(weights, [r['precision'] for r in results_decision], 's--', linewidth=1.5, markersize=4, 
             color='green', alpha=0.7, label='Precision')
    ax2.plot(weights, [r['recall'] for r in results_decision], '^--', linewidth=1.5, markersize=4, 
             color='red', alpha=0.7, label='Recall')
    ax2.plot(weights, [r['specificity'] for r in results_decision], 'd--', linewidth=1.5, markersize=4, 
             color='orange', alpha=0.7, label='Specificity')
    ax2.axvline(best_weight_decision, color='purple', linestyle='--', linewidth=2, 
                label=f'Best: {best_weight_decision:.2f}')
    ax2.set_xlabel('Audio Weight (Text = 1 - Audio)', fontsize=12)
    ax2.set_ylabel('Score', fontsize=12)
    ax2.set_title('Method 2: Decision Fusion (Threshold-Aware)', fontsize=14, fontweight='bold')
    ax2.legend()
    ax2.grid(alpha=0.3)
    
    plt.tight_layout()
    save_path = os.path.join(BASE_PATH, "late_fusion_optimization_corrected.png")
    plt.savefig(save_path, dpi=150, bbox_inches='tight')
    plt.close()
    print(f"\nSaved: {save_path}")
    
    return {
        'prob': {'weight': best_weight_prob, 'metrics': results_prob[best_idx_prob]},
        'decision': {'weight': best_weight_decision, 'metrics': results_decision[best_idx_decision]}
    }


def evaluate_on_test(test_data, best_configs, audio_threshold=0.3, text_threshold=0.2):
    """Evaluate best configurations on test set"""
    
    print(f"\n{'='*70}")
    print(f"Test Set Evaluation")
    print(f"{'='*70}\n")
    
    for method_name, config in best_configs.items():
        weight = config['weight']
        metrics = evaluate_late_fusion(test_data, weight, audio_threshold, text_threshold)
        
        # Get the correct method metrics
        if method_name == 'prob':
            test_metrics = metrics['prob']
        else:
            test_metrics = metrics['decision']
        
        print(f"[{method_name.upper()}] Audio Weight: {weight:.2f}")
        print(f"  Test F1:          {test_metrics['f1']:.4f}")
        print(f"  Test Precision:   {test_metrics['precision']:.4f}")
        print(f"  Test Recall:      {test_metrics['recall']:.4f}")
        print(f"  Test Specificity: {test_metrics['specificity']:.4f}\n")
    
    # Baseline comparison
    print(f"[Baseline Comparison]")
    
    # Audio only
    audio_only = evaluate_late_fusion(test_data, 1.0, audio_threshold, text_threshold)
    print(f"  Audio Only:")
    print(f"    F1:          {audio_only['decision']['f1']:.4f}")
    print(f"    Precision:   {audio_only['decision']['precision']:.4f}")
    print(f"    Recall:      {audio_only['decision']['recall']:.4f}")
    print(f"    Specificity: {audio_only['decision']['specificity']:.4f}")
    
    # Text only
    text_only = evaluate_late_fusion(test_data, 0.0, audio_threshold, text_threshold)
    print(f"\n  Text Only:")
    print(f"    F1:          {text_only['decision']['f1']:.4f}")
    print(f"    Precision:   {text_only['decision']['precision']:.4f}")
    print(f"    Recall:      {text_only['decision']['recall']:.4f}")
    print(f"    Specificity: {text_only['decision']['specificity']:.4f}")


# =============================================================================
# Main Execution
# =============================================================================
if __name__ == "__main__":
    print(f"{'='*70}")
    print(f"Threshold-Aware Late Fusion Analysis")
    print(f"{'='*70}\n")
    
    # Load models
    print("Loading audio model...")
    audio_checkpoint = torch.load(AUDIO_MODEL_PATH)
    audio_model = TransformerDepressionModel(
        input_dim=801, d_model=256, nhead=8, num_encoder_layers=3,
        dim_feedforward=512, dropout=0.3, q_type_vocab_size=5, q_type_embed_dim=32
    ).to(DEVICE)
    audio_model.load_state_dict(audio_checkpoint['model_state_dict'])
    audio_model.eval()
    print("Audio model loaded")
    
    print("Loading text model...")
    text_checkpoint = torch.load(TEXT_MODEL_PATH)
    text_model = TransformerTextDepressionModel(
        input_dim=801, d_model=256, nhead=8, num_encoder_layers=3,
        dim_feedforward=512, dropout=0.3, q_type_vocab_size=5, q_type_embed_dim=32
    ).to(DEVICE)
    text_model.load_state_dict(text_checkpoint['model_state_dict'])
    text_model.eval()
    print("Text model loaded")
    
    # Load datasets
    print("\nLoading datasets...")
    with open(AUDIO_DATA_PATH, 'rb') as f:
        audio_data = pickle.load(f)
    with open(TEXT_DATA_PATH, 'rb') as f:
        text_data = pickle.load(f)
    print(f"Audio data: {len(audio_data)} participants")
    print(f"Text data: {len(text_data)} participants")
    
    # Extract predictions
    print("\nExtracting predictions from both models...")
    all_results = extract_predictions_from_models(audio_model, text_model, audio_data, text_data)
    print(f"Extracted predictions for {len(all_results)} participants")
    
    # Split data by metadata groups
    print(f"\nSplitting data by metadata groups...")
    train_data, val_data, test_data = split_data_by_metadata_group(all_results, BASE_PATH)
    
    # Find optimal fusion weights
    best_configs = find_optimal_fusion_weight(train_data, val_data, AUDIO_THRESHOLD, TEXT_THRESHOLD)
    
    # Evaluate on test set
    evaluate_on_test(test_data, best_configs, AUDIO_THRESHOLD, TEXT_THRESHOLD)
    
    print(f"\n{'='*70}")
    print(f"Analysis Complete!")
    print(f"{'='*70}")

Threshold-Aware Late Fusion Analysis

Loading audio model...
Audio model loaded
Loading text model...
Text model loaded

Loading datasets...
Audio data: 186 participants
Text data: 186 participants

Extracting predictions from both models...
Extracted predictions for 186 participants

Splitting data by metadata groups...

Metadata-based split:
  Train: 107 (Normal: 76, Depression: 31)
  Val:   33 (Normal: 21, Depression: 12)
  Test:  46 (Normal: 32, Depression: 14)

Late Fusion Weight Optimization (Threshold-Aware)
Audio Threshold: 0.68
Text Threshold:  0.2

[Method 1: Probability Fusion]
  Best Audio Weight: 0.00
  Best F1:           0.0000
  Precision:         0.0000
  Recall:            0.0000
  Specificity:       1.0000

[Method 2: Decision Fusion (Threshold-Aware)]
  Best Audio Weight: 0.00
  Best F1:           0.5333
  Precision:         0.3636
  Recall:            1.0000
  Specificity:       0.0000

Saved: D:\depression_dataset(DAIC-WOZ)\late_fusion_optimization_corrected.png

T